# 01 — International Food Database Analysis

## USDA FoodData Central: Foundation Foods and FNDDS

This notebook completes the USDA part of the CeylonSavour international food-database analysis.

It:

- discovers the project and release folders safely;
- extracts USDA archives when required;
- inventories files and schemas consistently;
- identifies dataset-specific tables and keys;
- restricts Foundation analysis to actual Foundation Food records;
- resolves the FNDDS nutrient identifier correctly;
- performs schema and value-level relationship validation;
- analyses categories, missing values, duplicates, and nutrient completeness;
- creates a dataset-specific nutrient map;
- generates a normalized USDA reference table;
- exports a complete, internally consistent result set and manifest.

The next notebook section will begin with FlavorDB.

## 0. Configuration

Only change `PROJECT_ROOT_OVERRIDE` when automatic project discovery cannot find the repository.
Normally it should remain `None`.

In [4]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import json
import re
import warnings
import zipfile

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)

PROJECT_ROOT_OVERRIDE = None
AUTO_EXTRACT_ARCHIVES = True
CLEAN_PREVIOUS_USDA_OUTPUTS = True

RELATIONSHIP_VALID_THRESHOLD = 95.0
RELIABLE_MIN_COMPLETENESS = 50.0
QUALITY_MIN_SELECTED_NUTRIENTS = 4
NORMALIZED_DUPLICATE_AGGREGATION = "mean"

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

## 1. Project-root and USDA path discovery

In [5]:
def _candidate_project_roots():
    candidates = []

    if PROJECT_ROOT_OVERRIDE:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    home = Path.home()
    candidates.extend([
        home / "Github" / "Ceylon-Savour",
        home / "GitHub" / "Ceylon-Savour",
        Path("C:/Github/Ceylon-Savour"),
        Path("C:/GitHub/Ceylon-Savour"),
        Path("D:/Github/Ceylon-Savour"),
        Path("D:/GitHub/Ceylon-Savour"),
    ])

    unique = []
    seen = set()

    for candidate in candidates:
        try:
            resolved = candidate.expanduser().resolve()
        except Exception:
            continue

        key = str(resolved).lower()
        if key not in seen:
            unique.append(resolved)
            seen.add(key)

    return unique


def find_project_root():
    candidates = _candidate_project_roots()

    # Strongest signal: the USDA raw-data directory exists.
    for candidate in candidates:
        if (
            candidate / "data" / "raw" / "international" / "usda"
        ).exists():
            return candidate, "usda_data_directory"

    # Repository-structure fallback.
    for candidate in candidates:
        if (
            (candidate / ".git").exists()
            or (
                (candidate / "data").exists()
                and (candidate / "notebooks").exists()
            )
        ):
            return candidate, "repository_structure"

    # Non-failing fallback. The readiness checks later will explain missing data.
    return Path.cwd().resolve(), "current_directory_fallback"


PROJECT_ROOT, PROJECT_ROOT_METHOD = find_project_root()

USDA_ROOT = (
    PROJECT_ROOT / "data" / "raw" / "international" / "usda"
)
PROCESSED_USDA_ROOT = (
    PROJECT_ROOT / "data" / "processed" / "international" / "usda"
)
PROCESSED_USDA_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Discovery method:", PROJECT_ROOT_METHOD)
print("USDA raw root:", USDA_ROOT)
print("Processed output root:", PROCESSED_USDA_ROOT)

Project root: C:\Github\Ceylon-Savour
Discovery method: usda_data_directory
USDA raw root: C:\Github\Ceylon-Savour\data\raw\international\usda
Processed output root: C:\Github\Ceylon-Savour\data\processed\international\usda


## 2. Release discovery and archive extraction

In [6]:
def release_sort_key(path):
    numbers = [
        int(value)
        for value in re.findall(r"\d+", path.name)
    ]
    return tuple(numbers) if numbers else (0,)


def find_latest_release(dataset_root):
    if not dataset_root.exists():
        return None

    releases = sorted(
        [
            path
            for path in dataset_root.glob("release_*")
            if path.is_dir()
        ],
        key=release_sort_key,
    )

    return releases[-1] if releases else None


def ensure_extracted(release_path):
    if release_path is None:
        return {
            "release": None,
            "archive": None,
            "extracted": None,
            "status": "release_not_found",
        }

    extracted = release_path / "extracted"
    archive_dir = release_path / "archive"
    extracted.mkdir(parents=True, exist_ok=True)

    existing_csvs = list(extracted.rglob("*.csv"))
    if existing_csvs:
        return {
            "release": release_path,
            "archive": None,
            "extracted": extracted,
            "status": "ready",
        }

    archives = []
    if archive_dir.exists():
        archives.extend(sorted(archive_dir.rglob("*.zip")))
    archives.extend(sorted(release_path.glob("*.zip")))

    # Remove duplicate paths while preserving order.
    archives = list(dict.fromkeys(path.resolve() for path in archives))

    if not archives:
        return {
            "release": release_path,
            "archive": None,
            "extracted": extracted,
            "status": "archive_not_found",
        }

    archive_path = archives[0]

    if not AUTO_EXTRACT_ARCHIVES:
        return {
            "release": release_path,
            "archive": archive_path,
            "extracted": extracted,
            "status": "archive_found_not_extracted",
        }

    try:
        with zipfile.ZipFile(archive_path, "r") as zip_file:
            zip_file.extractall(extracted)
        status = "extracted"
    except Exception as error:
        status = f"extraction_failed: {error}"

    return {
        "release": release_path,
        "archive": archive_path,
        "extracted": extracted,
        "status": status,
    }


FOUNDATION_RELEASE = find_latest_release(USDA_ROOT / "foundation")
FNDDS_RELEASE = find_latest_release(USDA_ROOT / "fndds")

DATASET_PATHS = {
    "foundation": ensure_extracted(FOUNDATION_RELEASE),
    "fndds": ensure_extracted(FNDDS_RELEASE),
}

release_status_rows = []

for dataset, details in DATASET_PATHS.items():
    extracted = details["extracted"]
    extracted_csv_count = (
        len(list(extracted.rglob("*.csv")))
        if extracted is not None and extracted.exists()
        else 0
    )

    release_status_rows.append({
        "dataset": dataset,
        "dataset_label": (
            "Foundation Foods"
            if dataset == "foundation"
            else "FNDDS"
        ),
        "release_path": (
            str(details["release"])
            if details["release"] is not None
            else None
        ),
        "archive_path": (
            str(details["archive"])
            if details["archive"] is not None
            else None
        ),
        "extracted_path": (
            str(extracted)
            if extracted is not None
            else None
        ),
        "extracted_csv_count": extracted_csv_count,
        "status": details["status"],
        "ready": extracted_csv_count > 0,
    })

release_status_df = pd.DataFrame(release_status_rows)
USDA_READY = bool(
    not release_status_df.empty
    and release_status_df["ready"].all()
)

display(release_status_df)

if USDA_READY:
    print("Both USDA releases are ready.")
else:
    print(
        "USDA files are not fully ready. "
        "The notebook will continue without a path error, "
        "but validation will fail until the releases are available."
    )

,dataset,dataset_label,release_path,archive_path,extracted_path,extracted_csv_count,status,ready
0,foundation,Foundation Foods,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04,None,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted,24,ready,True
1,fndds,FNDDS,C:\Github\Ceylon-Savour\data\raw\international\usda\fndds\release_2024_10,None,C:\Github\Ceylon-Savour\data\raw\international\usda\fndds\release_2024_10\extracted,12,ready,True


Both USDA releases are ready.


## 3. Raw file inventory and consistent schema inventory

The schema inventory intentionally includes only CSV files inside each release's
`extracted` directory. Checksum CSV files in archive folders remain in the raw file
inventory but are excluded from both schema outputs.

In [7]:
FILE_INVENTORY_COLUMNS = [
    "dataset", "dataset_label", "release", "storage_area",
    "filename", "extension", "size_bytes", "size_mb",
    "relative_path",
]


def classify_storage_area(file_path, release_path):
    try:
        relative_parts = [
            part.lower()
            for part in file_path.relative_to(release_path).parts
        ]
    except Exception:
        relative_parts = [part.lower() for part in file_path.parts]

    if "extracted" in relative_parts:
        return "extracted"
    if "archive" in relative_parts:
        return "archive"
    if "checksums" in relative_parts or "checksum" in file_path.name.lower():
        return "checksums"
    return "other"


def build_file_inventory():
    rows = []

    for dataset, details in DATASET_PATHS.items():
        release = details["release"]
        if release is None or not release.exists():
            continue

        for file_path in sorted(release.rglob("*")):
            if not file_path.is_file():
                continue

            try:
                relative_path = file_path.relative_to(PROJECT_ROOT)
            except ValueError:
                relative_path = file_path

            rows.append({
                "dataset": dataset,
                "dataset_label": (
                    "Foundation Foods"
                    if dataset == "foundation"
                    else "FNDDS"
                ),
                "release": release.name,
                "storage_area": classify_storage_area(
                    file_path,
                    release,
                ),
                "filename": file_path.name,
                "extension": file_path.suffix.lower(),
                "size_bytes": file_path.stat().st_size,
                "size_mb": round(
                    file_path.stat().st_size / (1024 ** 2),
                    4,
                ),
                "relative_path": str(relative_path),
            })

    return pd.DataFrame(rows, columns=FILE_INVENTORY_COLUMNS)


file_inventory_df = build_file_inventory()

if file_inventory_df.empty:
    inventory_summary_df = pd.DataFrame(
        columns=[
            "dataset", "storage_area", "extension",
            "file_count", "total_size_mb",
        ]
    )
else:
    inventory_summary_df = (
        file_inventory_df
        .groupby(
            ["dataset", "storage_area", "extension"],
            dropna=False,
        )
        .agg(
            file_count=("filename", "count"),
            total_size_mb=("size_mb", "sum"),
        )
        .reset_index()
    )
    inventory_summary_df["total_size_mb"] = (
        inventory_summary_df["total_size_mb"].round(4)
    )

display(inventory_summary_df)

,dataset,storage_area,extension,file_count,total_size_mb
0,fndds,archive,,1,0.0000
1,fndds,archive,.dvc,1,0.0001
2,fndds,archive,.zip,1,3.1716
3,fndds,checksums,.csv,1,0.0002
4,fndds,extracted,,1,0.0000
5,fndds,extracted,.csv,12,24.2930
6,fndds,extracted,.xlsx,1,0.0301
7,foundation,archive,,1,0.0001
8,foundation,archive,.dvc,1,0.0001
9,foundation,archive,.zip,1,3.6483


In [8]:
def read_csv_header_flexible(path):
    encodings = ["utf-8-sig", "utf-8", "latin-1"]
    last_error = None

    for encoding in encodings:
        try:
            header = pd.read_csv(
                path,
                nrows=0,
                encoding=encoding,
            )
            return {
                "columns": header.columns.tolist(),
                "column_count": len(header.columns),
                "encoding": encoding,
                "error": None,
            }
        except Exception as error:
            last_error = error

    return {
        "columns": [],
        "column_count": 0,
        "encoding": None,
        "error": str(last_error),
    }


schema_rows = []

for dataset, details in DATASET_PATHS.items():
    extracted = details["extracted"]
    if extracted is None or not extracted.exists():
        continue

    for csv_path in sorted(extracted.rglob("*.csv")):
        result = read_csv_header_flexible(csv_path)

        try:
            relative_path = csv_path.relative_to(PROJECT_ROOT)
        except ValueError:
            relative_path = csv_path

        schema_rows.append({
            "dataset": dataset,
            "dataset_label": (
                "Foundation Foods"
                if dataset == "foundation"
                else "FNDDS"
            ),
            "scope": "extracted_data_csv",
            "filename": csv_path.name,
            "relative_path": str(relative_path),
            "size_mb": round(
                csv_path.stat().st_size / (1024 ** 2),
                4,
            ),
            "column_count": result["column_count"],
            "encoding": result["encoding"],
            "columns": result["columns"],
            "error": result["error"],
        })

schema_file_summary_df = pd.DataFrame(
    schema_rows,
    columns=[
        "dataset", "dataset_label", "scope", "filename",
        "relative_path", "size_mb", "column_count",
        "encoding", "columns", "error",
    ],
)

if schema_file_summary_df.empty:
    schema_long_df = pd.DataFrame(
        columns=[
            "dataset", "dataset_label", "scope", "filename",
            "column_position", "column_name",
        ]
    )
else:
    schema_long_df = (
        schema_file_summary_df[
            [
                "dataset", "dataset_label", "scope",
                "filename", "columns",
            ]
        ]
        .explode("columns")
        .rename(columns={"columns": "column_name"})
        .reset_index(drop=True)
    )
    schema_long_df["column_position"] = (
        schema_long_df
        .groupby(["dataset", "filename"])
        .cumcount()
        .add(1)
    )
    schema_long_df = schema_long_df[
        [
            "dataset", "dataset_label", "scope",
            "filename", "column_position", "column_name",
        ]
    ]

display(
    schema_file_summary_df[
        [
            "dataset", "scope", "filename",
            "column_count", "error",
        ]
    ]
)

,dataset,scope,filename,column_count,error
0,foundation,extracted_data_csv,acquisition_samples.csv,2,None
1,foundation,extracted_data_csv,agricultural_samples.csv,5,None
2,foundation,extracted_data_csv,food.csv,5,None
3,foundation,extracted_data_csv,food_attribute.csv,6,None
4,foundation,extracted_data_csv,food_attribute_type.csv,3,None
5,foundation,extracted_data_csv,food_calorie_conversion_factor.csv,4,None
6,foundation,extracted_data_csv,food_category.csv,3,None
7,foundation,extracted_data_csv,food_component.csv,8,None
8,foundation,extracted_data_csv,food_nutrient.csv,11,None
9,foundation,extracted_data_csv,food_nutrient_conversion_factor.csv,2,None


## 4. Exact table-role mapping

`input_food.csv` is included for FNDDS so that its international ingredient
references are not overlooked.

In [9]:
PREFERRED_TABLES = {
    "foundation": {
        "food": "food.csv",
        "foundation_food": "foundation_food.csv",
        "nutrient": "nutrient.csv",
        "food_nutrient": "food_nutrient.csv",
        "food_category": "food_category.csv",
        "food_portion": "food_portion.csv",
        "measure_unit": "measure_unit.csv",
        "input_food": "input_food.csv",
    },
    "fndds": {
        "food": "food.csv",
        "nutrient": "nutrient.csv",
        "food_nutrient": "food_nutrient.csv",
        "food_portion": "food_portion.csv",
        "measure_unit": "measure_unit.csv",
        "survey_fndds_food": "survey_fndds_food.csv",
        "wweia_food_category": "wweia_food_category.csv",
        "input_food": "input_food.csv",
    },
}


def find_exact_table(dataset, filename):
    extracted = DATASET_PATHS.get(dataset, {}).get("extracted")
    if extracted is None or not extracted.exists():
        return None

    matches = [
        path
        for path in extracted.rglob(filename)
        if path.is_file()
    ]

    if not matches:
        return None

    return sorted(matches)[0]


table_paths = {}
mapping_rows = []

for dataset, role_mapping in PREFERRED_TABLES.items():
    table_paths[dataset] = {}

    for role, filename in role_mapping.items():
        path = find_exact_table(dataset, filename)
        table_paths[dataset][role] = path

        schema_match = schema_file_summary_df[
            schema_file_summary_df["dataset"].eq(dataset)
            & schema_file_summary_df["filename"].str.lower().eq(
                filename.lower()
            )
        ]

        columns = (
            schema_match.iloc[0]["columns"]
            if not schema_match.empty
            else []
        )

        mapping_rows.append({
            "dataset": dataset,
            "dataset_label": (
                "Foundation Foods"
                if dataset == "foundation"
                else "FNDDS"
            ),
            "role": role,
            "expected_filename": filename,
            "status": "available" if path is not None else "not_available",
            "path": str(path) if path is not None else None,
            "column_count": len(columns),
            "columns": columns,
        })

table_role_mapping_df = pd.DataFrame(mapping_rows)
display(table_role_mapping_df)

,dataset,dataset_label,role,expected_filename,status,path,column_count,columns
0,foundation,Foundation Foods,food,food.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\food.csv,5,"[fdc_id, data_type, description, food_category_id, publication_date]"
1,foundation,Foundation Foods,foundation_food,foundation_food.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\foundation_food.csv,3,"[fdc_id, NDB_number, footnote]"
2,foundation,Foundation Foods,nutrient,nutrient.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\nutrient.csv,5,"[id, name, unit_name, nutrient_nbr, rank]"
3,foundation,Foundation Foods,food_nutrient,food_nutrient.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\food_nutrient.csv,11,"[id, fdc_id, nutrient_id, amount, data_points, derivation_id, min, max, median, footnote, min_year_acquired]"
4,foundation,Foundation Foods,food_category,food_category.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\food_category.csv,3,"[id, code, description]"
5,foundation,Foundation Foods,food_portion,food_portion.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\food_portion.csv,11,"[id, fdc_id, seq_num, amount, measure_unit_id, portion_description, modifier, gram_weight, data_points, footnote, min_year_acquired]"
6,foundation,Foundation Foods,measure_unit,measure_unit.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\measure_unit.csv,2,"[id, name]"
7,foundation,Foundation Foods,input_food,input_food.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\foundation\release_2026_04\extracted\input_food.csv,12,"[id, fdc_id, fdc_of_input_food, seq_num, amount, ingredient_code, ingredient_description, unit, portion_code, portion_description, gram_weight, retention_code]"
8,fndds,FNDDS,food,food.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\fndds\release_2024_10\extracted\food.csv,5,"[fdc_id, data_type, description, food_category_id, publication_date]"
9,fndds,FNDDS,nutrient,nutrient.csv,available,C:\Github\Ceylon-Savour\data\raw\international\usda\fndds\release_2024_10\extracted\nutrient.csv,5,"[id, name, unit_name, nutrient_nbr, rank]"


## 5. Safe table loading

In [10]:
def read_csv_flexible(path, preferred_columns=None):
    if path is None or not Path(path).exists():
        return pd.DataFrame()

    header_result = read_csv_header_flexible(Path(path))
    available_columns = header_result["columns"]

    usecols = None
    if preferred_columns is not None:
        usecols = [
            column
            for column in preferred_columns
            if column in available_columns
        ]

    encodings = ["utf-8-sig", "utf-8", "latin-1"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                usecols=usecols,
                encoding=encoding,
                low_memory=False,
            )
        except Exception as error:
            last_error = error

    warnings.warn(
        f"Could not load {path}: {last_error}"
    )
    return pd.DataFrame(columns=usecols or available_columns)


def load_role(dataset, role, preferred_columns=None):
    return read_csv_flexible(
        table_paths.get(dataset, {}).get(role),
        preferred_columns=preferred_columns,
    )


def to_nullable_integer(df, columns):
    result = df.copy()
    for column in columns:
        if column in result.columns:
            numeric = pd.to_numeric(
                result[column],
                errors="coerce",
            )
            if (numeric.dropna() % 1 == 0).all():
                result[column] = numeric.astype("Int64")
            else:
                result[column] = numeric
    return result


def to_numeric_columns(df, columns):
    result = df.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="coerce",
            )
    return result


foundation_food_all = load_role(
    "foundation",
    "food",
    [
        "fdc_id", "data_type", "description",
        "food_category_id", "publication_date",
    ],
)
foundation_food_detail = load_role(
    "foundation",
    "foundation_food",
)
foundation_nutrient = load_role(
    "foundation",
    "nutrient",
    ["id", "name", "unit_name", "nutrient_nbr", "rank"],
)
foundation_food_nutrient_raw = load_role(
    "foundation",
    "food_nutrient",
    [
        "id", "fdc_id", "nutrient_id", "amount",
        "data_points", "derivation_id", "min",
        "max", "median",
    ],
)
foundation_food_category = load_role(
    "foundation",
    "food_category",
    ["id", "code", "description"],
)
foundation_food_portion_raw = load_role(
    "foundation",
    "food_portion",
)
foundation_measure_unit = load_role(
    "foundation",
    "measure_unit",
    ["id", "name"],
)
foundation_input_food = load_role(
    "foundation",
    "input_food",
)

fndds_food = load_role(
    "fndds",
    "food",
    [
        "fdc_id", "data_type", "description",
        "food_category_id", "publication_date",
    ],
)
fndds_nutrient = load_role(
    "fndds",
    "nutrient",
    ["id", "name", "unit_name", "nutrient_nbr", "rank"],
)
fndds_food_nutrient_raw = load_role(
    "fndds",
    "food_nutrient",
    [
        "id", "fdc_id", "nutrient_id", "amount",
        "data_points", "derivation_id", "min",
        "max", "median",
    ],
)
fndds_food_portion = load_role(
    "fndds",
    "food_portion",
)
fndds_measure_unit = load_role(
    "fndds",
    "measure_unit",
    ["id", "name"],
)
fndds_survey = load_role(
    "fndds",
    "survey_fndds_food",
)
fndds_wweia_category = load_role(
    "fndds",
    "wweia_food_category",
)
fndds_input_food = load_role(
    "fndds",
    "input_food",
)

for dataframe_name in [
    "foundation_food_all", "foundation_food_detail",
    "foundation_nutrient", "foundation_food_nutrient_raw",
    "foundation_food_category", "foundation_food_portion_raw",
    "foundation_measure_unit", "foundation_input_food",
    "fndds_food", "fndds_nutrient",
    "fndds_food_nutrient_raw", "fndds_food_portion",
    "fndds_measure_unit", "fndds_survey",
    "fndds_wweia_category", "fndds_input_food",
]:
    dataframe = globals()[dataframe_name]
    dataframe = to_nullable_integer(
        dataframe,
        [
            "id", "fdc_id", "nutrient_id", "nutrient_nbr",
            "food_category_id", "measure_unit_id",
            "wweia_category_number", "wweia_food_category",
            "fdc_id_of_input_food",
        ],
    )
    dataframe = to_numeric_columns(
        dataframe,
        ["amount", "gram_weight", "data_points", "min", "max", "median"],
    )
    globals()[dataframe_name] = dataframe

print("Core USDA tables loaded.")

Core USDA tables loaded.


## 6. Select actual Foundation Foods

The generic Foundation release `food.csv` can contain Foundation Foods,
sample foods, acquisition foods, and sub-sample foods. The analysis therefore
selects the Foundation subtype using `foundation_food.csv` first and uses
`food.data_type == "foundation_food"` only as a fallback.

In [11]:
def select_final_foundation_foods(food_all, foundation_detail):
    if (
        not food_all.empty
        and "fdc_id" in food_all.columns
        and not foundation_detail.empty
        and "fdc_id" in foundation_detail.columns
    ):
        selected_ids = (
            foundation_detail["fdc_id"]
            .dropna()
            .drop_duplicates()
        )

        selected = food_all[
            food_all["fdc_id"].isin(selected_ids)
        ].copy()

        return selected.drop_duplicates("fdc_id"), (
            "food.csv INNER JOIN foundation_food.csv ON fdc_id"
        )

    if (
        not food_all.empty
        and "data_type" in food_all.columns
    ):
        selected = food_all[
            food_all["data_type"]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("foundation_food")
        ].copy()

        return selected.drop_duplicates("fdc_id"), (
            "food.data_type == foundation_food fallback"
        )

    return pd.DataFrame(columns=food_all.columns), (
        "foundation_food_selection_unavailable"
    )


foundation_food, FOUNDATION_FILTER_METHOD = (
    select_final_foundation_foods(
        foundation_food_all,
        foundation_food_detail,
    )
)

foundation_ids = set(
    foundation_food.get(
        "fdc_id",
        pd.Series(dtype="Int64"),
    ).dropna().tolist()
)

fndds_ids = set(
    fndds_food.get(
        "fdc_id",
        pd.Series(dtype="Int64"),
    ).dropna().tolist()
)

foundation_food_nutrient = (
    foundation_food_nutrient_raw[
        foundation_food_nutrient_raw["fdc_id"].isin(
            foundation_ids
        )
    ].copy()
    if (
        not foundation_food_nutrient_raw.empty
        and "fdc_id" in foundation_food_nutrient_raw.columns
    )
    else pd.DataFrame(
        columns=foundation_food_nutrient_raw.columns
    )
)

foundation_food_portion = (
    foundation_food_portion_raw[
        foundation_food_portion_raw["fdc_id"].isin(
            foundation_ids
        )
    ].copy()
    if (
        not foundation_food_portion_raw.empty
        and "fdc_id" in foundation_food_portion_raw.columns
    )
    else pd.DataFrame(
        columns=foundation_food_portion_raw.columns
    )
)

fndds_food_nutrient = (
    fndds_food_nutrient_raw[
        fndds_food_nutrient_raw["fdc_id"].isin(
            fndds_ids
        )
    ].copy()
    if (
        not fndds_food_nutrient_raw.empty
        and "fdc_id" in fndds_food_nutrient_raw.columns
    )
    else pd.DataFrame(
        columns=fndds_food_nutrient_raw.columns
    )
)

if (
    not foundation_food_all.empty
    and "data_type" in foundation_food_all.columns
):
    foundation_record_type_summary_df = (
        foundation_food_all["data_type"]
        .fillna("<missing>")
        .value_counts(dropna=False)
        .rename_axis("data_type")
        .reset_index(name="row_count")
    )
else:
    foundation_record_type_summary_df = pd.DataFrame(
        columns=["data_type", "row_count"]
    )

print("Foundation selection method:", FOUNDATION_FILTER_METHOD)
print("Generic Foundation food.csv rows:", len(foundation_food_all))
print("Selected final Foundation Food rows:", len(foundation_food))
display(foundation_record_type_summary_df)

Foundation selection method: food.csv INNER JOIN foundation_food.csv ON fdc_id
Generic Foundation food.csv rows: 87990
Selected final Foundation Food rows: 395


,data_type,row_count
0,sub_sample_food,75055
1,market_acquisition,7577
2,sample_food,4079
3,agricultural_acquisition,810
4,foundation_food,469


## 7. Resolve dataset-specific nutrient identifiers

The release is tested against both `nutrient.id` and `nutrient.nutrient_nbr`.
The key with the highest actual value match is selected for each dataset.
For the current FNDDS release this should resolve to `nutrient_nbr`.

In [12]:
def identifier_match_stats(food_nutrient_df, nutrient_df, right_key):
    if (
        food_nutrient_df.empty
        or nutrient_df.empty
        or "nutrient_id" not in food_nutrient_df.columns
        or right_key not in nutrient_df.columns
    ):
        return {
            "non_null_rows": 0,
            "matched_rows": 0,
            "unmatched_rows": 0,
            "match_percent": 0.0,
        }

    left = pd.to_numeric(
        food_nutrient_df["nutrient_id"],
        errors="coerce",
    )
    right_values = set(
        pd.to_numeric(
            nutrient_df[right_key],
            errors="coerce",
        ).dropna().tolist()
    )

    non_null = left.dropna()
    matched_mask = non_null.isin(right_values)
    matched_rows = int(matched_mask.sum())
    non_null_rows = int(len(non_null))

    return {
        "non_null_rows": non_null_rows,
        "matched_rows": matched_rows,
        "unmatched_rows": non_null_rows - matched_rows,
        "match_percent": round(
            matched_rows / non_null_rows * 100,
            4,
        ) if non_null_rows else 0.0,
    }


resolution_rows = []

for dataset, food_nutrient_df, nutrient_df in [
    (
        "foundation",
        foundation_food_nutrient,
        foundation_nutrient,
    ),
    (
        "fndds",
        fndds_food_nutrient,
        fndds_nutrient,
    ),
]:
    for candidate_key in ["id", "nutrient_nbr"]:
        stats = identifier_match_stats(
            food_nutrient_df,
            nutrient_df,
            candidate_key,
        )
        resolution_rows.append({
            "dataset": dataset,
            "candidate_nutrient_key": candidate_key,
            **stats,
        })

nutrient_key_resolution_df = pd.DataFrame(resolution_rows)

def choose_nutrient_key(dataset):
    candidates = nutrient_key_resolution_df[
        nutrient_key_resolution_df["dataset"].eq(dataset)
    ].copy()

    if candidates.empty:
        return "id"

    # On ties, Foundation prefers id and FNDDS prefers nutrient_nbr.
    preference = (
        ["nutrient_nbr", "id"]
        if dataset == "fndds"
        else ["id", "nutrient_nbr"]
    )
    preference_rank = {
        key: len(preference) - index
        for index, key in enumerate(preference)
    }
    candidates["preference_rank"] = (
        candidates["candidate_nutrient_key"]
        .map(preference_rank)
        .fillna(0)
    )

    selected = candidates.sort_values(
        ["match_percent", "preference_rank"],
        ascending=[False, False],
    ).iloc[0]

    return selected["candidate_nutrient_key"]


FOUNDATION_NUTRIENT_KEY = choose_nutrient_key("foundation")
FNDDS_NUTRIENT_KEY = choose_nutrient_key("fndds")

nutrient_key_resolution_df["selected"] = (
    (
        nutrient_key_resolution_df["dataset"].eq("foundation")
        & nutrient_key_resolution_df[
            "candidate_nutrient_key"
        ].eq(FOUNDATION_NUTRIENT_KEY)
    )
    |
    (
        nutrient_key_resolution_df["dataset"].eq("fndds")
        & nutrient_key_resolution_df[
            "candidate_nutrient_key"
        ].eq(FNDDS_NUTRIENT_KEY)
    )
)

display(nutrient_key_resolution_df)
print("Foundation nutrient key:", FOUNDATION_NUTRIENT_KEY)
print("FNDDS nutrient key:", FNDDS_NUTRIENT_KEY)

,dataset,candidate_nutrient_key,non_null_rows,matched_rows,unmatched_rows,match_percent,selected
0,foundation,id,17008,16975,33,99.806,True
1,foundation,nutrient_nbr,17008,0,17008,0.000,False
2,fndds,id,353015,0,353015,0.000,False
3,fndds,nutrient_nbr,353015,353015,0,100.000,True


Foundation nutrient key: id
FNDDS nutrient key: nutrient_nbr


## 8. Relationship schema and value validation

`relationship_valid` is based on actual matched values, not only column presence.

In [13]:
def validate_relationship(
    dataset,
    relationship,
    left_table_name,
    left_df,
    left_key,
    right_table_name,
    right_df,
    right_key,
    threshold_percent=RELATIONSHIP_VALID_THRESHOLD,
):
    left_key_exists = left_key in left_df.columns
    right_key_exists = right_key in right_df.columns
    schema_valid = left_key_exists and right_key_exists

    if not schema_valid:
        return {
            "dataset": dataset,
            "relationship": relationship,
            "left_table": left_table_name,
            "left_key": left_key,
            "left_key_exists": left_key_exists,
            "right_table": right_table_name,
            "right_key": right_key,
            "right_key_exists": right_key_exists,
            "schema_valid": False,
            "left_rows": len(left_df),
            "left_null_keys": None,
            "left_non_null_rows": 0,
            "matched_rows": 0,
            "unmatched_rows": 0,
            "match_percent": 0.0,
            "right_duplicate_keys": None,
            "threshold_percent": threshold_percent,
            "relationship_valid": False,
        }

    left_values = pd.to_numeric(
        left_df[left_key],
        errors="coerce",
    )
    right_values = pd.to_numeric(
        right_df[right_key],
        errors="coerce",
    )

    left_null_keys = int(left_values.isna().sum())
    left_non_null = left_values.dropna()
    right_non_null = right_values.dropna()

    right_set = set(right_non_null.tolist())
    matched_mask = left_non_null.isin(right_set)
    matched_rows = int(matched_mask.sum())
    left_non_null_rows = int(len(left_non_null))
    unmatched_rows = left_non_null_rows - matched_rows
    match_percent = (
        matched_rows / left_non_null_rows * 100
        if left_non_null_rows
        else 0.0
    )
    right_duplicate_keys = int(
        right_non_null.duplicated().sum()
    )

    relationship_valid = bool(
        schema_valid
        and match_percent >= threshold_percent
        and right_duplicate_keys == 0
    )

    return {
        "dataset": dataset,
        "relationship": relationship,
        "left_table": left_table_name,
        "left_key": left_key,
        "left_key_exists": left_key_exists,
        "right_table": right_table_name,
        "right_key": right_key,
        "right_key_exists": right_key_exists,
        "schema_valid": schema_valid,
        "left_rows": len(left_df),
        "left_null_keys": left_null_keys,
        "left_non_null_rows": left_non_null_rows,
        "matched_rows": matched_rows,
        "unmatched_rows": unmatched_rows,
        "match_percent": round(match_percent, 4),
        "right_duplicate_keys": right_duplicate_keys,
        "threshold_percent": threshold_percent,
        "relationship_valid": relationship_valid,
    }


relationship_definitions = [
    (
        "foundation", "food_nutrient to food",
        "food_nutrient.csv", foundation_food_nutrient, "fdc_id",
        "selected foundation food", foundation_food, "fdc_id",
    ),
    (
        "foundation", "food_nutrient to nutrient",
        "food_nutrient.csv", foundation_food_nutrient, "nutrient_id",
        "nutrient.csv", foundation_nutrient, FOUNDATION_NUTRIENT_KEY,
    ),
    (
        "foundation", "food_portion to food",
        "food_portion.csv", foundation_food_portion, "fdc_id",
        "selected foundation food", foundation_food, "fdc_id",
    ),
    (
        "foundation", "food_portion to measure_unit",
        "food_portion.csv", foundation_food_portion, "measure_unit_id",
        "measure_unit.csv", foundation_measure_unit, "id",
    ),
    (
        "foundation", "food to food_category",
        "selected foundation food", foundation_food, "food_category_id",
        "food_category.csv", foundation_food_category, "id",
    ),
    (
        "fndds", "food_nutrient to food",
        "food_nutrient.csv", fndds_food_nutrient, "fdc_id",
        "food.csv", fndds_food, "fdc_id",
    ),
    (
        "fndds", "food_nutrient to nutrient",
        "food_nutrient.csv", fndds_food_nutrient, "nutrient_id",
        "nutrient.csv", fndds_nutrient, FNDDS_NUTRIENT_KEY,
    ),
    (
        "fndds", "food_portion to food",
        "food_portion.csv", fndds_food_portion, "fdc_id",
        "food.csv", fndds_food, "fdc_id",
    ),
    (
        "fndds", "food_portion to measure_unit",
        "food_portion.csv", fndds_food_portion, "measure_unit_id",
        "measure_unit.csv", fndds_measure_unit, "id",
    ),
    (
        "fndds", "survey_fndds_food to food",
        "survey_fndds_food.csv", fndds_survey, "fdc_id",
        "food.csv", fndds_food, "fdc_id",
    ),
    (
        "fndds", "survey_fndds_food to WWEIA category",
        "survey_fndds_food.csv", fndds_survey, "wweia_category_number",
        "wweia_food_category.csv", fndds_wweia_category,
        "wweia_food_category",
    ),
]

relationship_validation_df = pd.DataFrame([
    validate_relationship(*definition)
    for definition in relationship_definitions
])

relationship_schema_validation_df = (
    relationship_validation_df[
        [
            "dataset", "relationship",
            "left_table", "left_key", "left_key_exists",
            "right_table", "right_key", "right_key_exists",
            "schema_valid",
        ]
    ].copy()
)

relationship_coverage_df = (
    relationship_validation_df[
        [
            "dataset", "relationship",
            "left_non_null_rows", "matched_rows",
            "unmatched_rows", "match_percent",
            "right_duplicate_keys",
            "threshold_percent", "relationship_valid",
        ]
    ].copy()
)

display(relationship_validation_df)

,dataset,relationship,left_table,left_key,left_key_exists,right_table,right_key,right_key_exists,schema_valid,left_rows,left_null_keys,left_non_null_rows,matched_rows,unmatched_rows,match_percent,right_duplicate_keys,threshold_percent,relationship_valid
0,foundation,food_nutrient to food,food_nutrient.csv,fdc_id,True,selected foundation food,fdc_id,True,True,17008,0,17008,17008,0,100.000,0,95.0,True
1,foundation,food_nutrient to nutrient,food_nutrient.csv,nutrient_id,True,nutrient.csv,id,True,True,17008,0,17008,16975,33,99.806,0,95.0,True
2,foundation,food_portion to food,food_portion.csv,fdc_id,True,selected foundation food,fdc_id,True,True,123,0,123,123,0,100.000,0,95.0,True
3,foundation,food_portion to measure_unit,food_portion.csv,measure_unit_id,True,measure_unit.csv,id,True,True,123,0,123,123,0,100.000,0,95.0,True
4,foundation,food to food_category,selected foundation food,food_category_id,True,food_category.csv,id,True,True,395,0,395,395,0,100.000,0,95.0,True
5,fndds,food_nutrient to food,food_nutrient.csv,fdc_id,True,food.csv,fdc_id,True,True,353015,0,353015,353015,0,100.000,0,95.0,True
6,fndds,food_nutrient to nutrient,food_nutrient.csv,nutrient_id,True,nutrient.csv,nutrient_nbr,True,True,353015,0,353015,353015,0,100.000,0,95.0,True
7,fndds,food_portion to food,food_portion.csv,fdc_id,True,food.csv,fdc_id,True,True,22046,0,22046,22046,0,100.000,0,95.0,True
8,fndds,food_portion to measure_unit,food_portion.csv,measure_unit_id,True,measure_unit.csv,id,True,True,22046,0,22046,22046,0,100.000,0,95.0,True
9,fndds,survey_fndds_food to food,survey_fndds_food.csv,fdc_id,True,food.csv,fdc_id,True,True,5432,0,5432,5432,0,100.000,0,95.0,True


## 9. Category joins and FNDDS ingredient-reference coverage

In [14]:
foundation_food_with_category = foundation_food.copy()
foundation_category_coverage = 0.0

if (
    not foundation_food.empty
    and "food_category_id" in foundation_food.columns
    and not foundation_food_category.empty
    and {"id", "description"}.issubset(
        foundation_food_category.columns
    )
):
    category_lookup = (
        foundation_food_category[
            ["id", "description"]
        ]
        .drop_duplicates("id")
        .rename(columns={
            "id": "food_category_id",
            "description": "food_category",
        })
    )

    foundation_food_with_category = (
        foundation_food
        .merge(
            category_lookup,
            on="food_category_id",
            how="left",
        )
    )

    foundation_category_coverage = round(
        foundation_food_with_category[
            "food_category"
        ].notna().mean() * 100,
        4,
    )


fndds_food_with_category = fndds_food.copy()
fndds_category_coverage = 0.0

if (
    not fndds_food.empty
    and not fndds_survey.empty
    and "fdc_id" in fndds_food.columns
    and {
        "fdc_id", "wweia_category_number",
    }.issubset(fndds_survey.columns)
):
    survey_lookup = (
        fndds_survey
        .drop_duplicates("fdc_id")
    )

    fndds_food_with_category = (
        fndds_food
        .merge(
            survey_lookup,
            on="fdc_id",
            how="left",
            suffixes=("", "_survey"),
        )
    )

    if (
        not fndds_wweia_category.empty
        and {
            "wweia_food_category",
            "wweia_food_category_description",
        }.issubset(fndds_wweia_category.columns)
    ):
        wweia_lookup = (
            fndds_wweia_category[
                [
                    "wweia_food_category",
                    "wweia_food_category_description",
                ]
            ]
            .drop_duplicates("wweia_food_category")
        )

        fndds_food_with_category = (
            fndds_food_with_category
            .merge(
                wweia_lookup,
                left_on="wweia_category_number",
                right_on="wweia_food_category",
                how="left",
            )
        )

        fndds_category_coverage = round(
            fndds_food_with_category[
                "wweia_food_category_description"
            ].notna().mean() * 100,
            4,
        )


category_strategy_df = pd.DataFrame([
    {
        "dataset": "foundation",
        "strategy": (
            "selected food.food_category_id -> food_category.id"
        ),
        "coverage_percent": foundation_category_coverage,
    },
    {
        "dataset": "fndds",
        "strategy": (
            "survey_fndds_food.wweia_category_number -> "
            "wweia_food_category.wweia_food_category"
        ),
        "coverage_percent": fndds_category_coverage,
    },
])

category_coverage_df = category_strategy_df.copy()
display(category_coverage_df)

,dataset,strategy,coverage_percent
0,foundation,selected food.food_category_id -> food_category.id,100.0
1,fndds,survey_fndds_food.wweia_category_number -> wweia_food_category.wweia_food_category,100.0


In [15]:
def first_existing_column(df, candidates):
    for column in candidates:
        if column in df.columns:
            return column
    return None


ingredient_description_column = first_existing_column(
    fndds_input_food,
    [
        "ingredient_description",
        "sr_description",
        "input_food_description",
        "description",
    ],
)

input_parent_key = first_existing_column(
    fndds_input_food,
    ["fdc_id", "food_fdc_id"],
)

foods_with_input_reference = 0
input_food_coverage_percent = 0.0
description_non_null_percent = 0.0

if (
    not fndds_input_food.empty
    and input_parent_key is not None
):
    referenced_ids = (
        pd.to_numeric(
            fndds_input_food[input_parent_key],
            errors="coerce",
        )
        .dropna()
        .unique()
    )
    foods_with_input_reference = len(referenced_ids)

    total_fndds_foods = (
        fndds_food["fdc_id"].dropna().nunique()
        if "fdc_id" in fndds_food.columns
        else 0
    )

    input_food_coverage_percent = round(
        foods_with_input_reference
        / total_fndds_foods
        * 100,
        4,
    ) if total_fndds_foods else 0.0

if (
    ingredient_description_column is not None
    and not fndds_input_food.empty
):
    description_non_null_percent = round(
        fndds_input_food[
            ingredient_description_column
        ].notna().mean() * 100,
        4,
    )

fndds_ingredient_reference_summary_df = pd.DataFrame([{
    "dataset": "fndds",
    "source_table": "input_food.csv",
    "row_count": len(fndds_input_food),
    "parent_food_key": input_parent_key,
    "ingredient_description_column": (
        ingredient_description_column
    ),
    "foods_with_input_reference": foods_with_input_reference,
    "fndds_food_coverage_percent": input_food_coverage_percent,
    "ingredient_description_non_null_percent": (
        description_non_null_percent
    ),
    "available_columns": ", ".join(
        map(str, fndds_input_food.columns.tolist())
    ),
    "interpretation": (
        "FNDDS provides ingredient references for many survey foods, "
        "but it is not a complete or authoritative ingredient source "
        "for Sri Lankan dishes."
    ),
}])

display(fndds_ingredient_reference_summary_df)

,dataset,source_table,row_count,parent_food_key,ingredient_description_column,foods_with_input_reference,fndds_food_coverage_percent,ingredient_description_non_null_percent,available_columns,interpretation
0,fndds,input_food.csv,18584,fdc_id,sr_description,5431,99.9816,100.0,"id, fdc_id, fdc_of_input_food, seq_num, amount, sr_code, sr_description, unit, portion_code, portion_description, gram_weight, retention_code","FNDDS provides ingredient references for many survey foods, but it is not a complete or authoritative ingredient source for Sri Lankan dishes."


## 10. Core table, join, missing-value, and duplicate summaries

In [16]:
def table_summary(
    dataset,
    role,
    df,
    source_description,
):
    return {
        "dataset": dataset,
        "role": role,
        "source_description": source_description,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_mb": round(
            df.memory_usage(deep=True).sum()
            / (1024 ** 2),
            4,
        ) if not df.empty else 0.0,
    }


table_summary_df = pd.DataFrame([
    table_summary(
        "foundation",
        "food_all",
        foundation_food_all,
        "generic food.csv including all record types",
    ),
    table_summary(
        "foundation",
        "food",
        foundation_food,
        "actual Foundation Foods selected by subtype table/filter",
    ),
    table_summary(
        "foundation",
        "food_nutrient",
        foundation_food_nutrient,
        "food_nutrient rows restricted to selected Foundation Foods",
    ),
    table_summary(
        "foundation",
        "nutrient_master",
        foundation_nutrient,
        "nutrient.csv master table",
    ),
    table_summary(
        "fndds",
        "food",
        fndds_food,
        "FNDDS food.csv",
    ),
    table_summary(
        "fndds",
        "food_nutrient",
        fndds_food_nutrient,
        "food_nutrient rows restricted to FNDDS food IDs",
    ),
    table_summary(
        "fndds",
        "nutrient_master",
        fndds_nutrient,
        "nutrient.csv master table",
    ),
    table_summary(
        "fndds",
        "input_food",
        fndds_input_food,
        "FNDDS ingredient/input-food references",
    ),
])

def relationship_percent(dataset, relationship):
    match = relationship_validation_df[
        relationship_validation_df["dataset"].eq(dataset)
        & relationship_validation_df[
            "relationship"
        ].eq(relationship)
    ]
    return (
        float(match.iloc[0]["match_percent"])
        if not match.empty
        else 0.0
    )


core_join_summary_df = pd.DataFrame([
    {
        "dataset": "foundation",
        "food_count": (
            foundation_food["fdc_id"].dropna().nunique()
            if "fdc_id" in foundation_food.columns
            else 0
        ),
        "food_nutrient_rows": len(foundation_food_nutrient),
        "nutrient_lookup_key": FOUNDATION_NUTRIENT_KEY,
        "food_relationship_match_percent": (
            relationship_percent(
                "foundation",
                "food_nutrient to food",
            )
        ),
        "nutrient_relationship_match_percent": (
            relationship_percent(
                "foundation",
                "food_nutrient to nutrient",
            )
        ),
        "category_coverage_percent": (
            foundation_category_coverage
        ),
    },
    {
        "dataset": "fndds",
        "food_count": (
            fndds_food["fdc_id"].dropna().nunique()
            if "fdc_id" in fndds_food.columns
            else 0
        ),
        "food_nutrient_rows": len(fndds_food_nutrient),
        "nutrient_lookup_key": FNDDS_NUTRIENT_KEY,
        "food_relationship_match_percent": (
            relationship_percent(
                "fndds",
                "food_nutrient to food",
            )
        ),
        "nutrient_relationship_match_percent": (
            relationship_percent(
                "fndds",
                "food_nutrient to nutrient",
            )
        ),
        "category_coverage_percent": (
            fndds_category_coverage
        ),
    },
])

display(table_summary_df)
display(core_join_summary_df)

,dataset,role,source_description,rows,columns,memory_mb
0,foundation,food_all,generic food.csv including all record types,87990,5,20.8028
1,foundation,food,actual Foundation Foods selected by subtype table/filter,395,5,0.0959
2,foundation,food_nutrient,food_nutrient rows restricted to selected Foundation Foods,17008,9,1.3463
3,foundation,nutrient_master,nutrient.csv master table,477,5,0.0709
4,fndds,food,FNDDS food.csv,5432,5,1.3027
5,fndds,food_nutrient,food_nutrient rows restricted to FNDDS food IDs,353015,9,25.2497
6,fndds,nutrient_master,nutrient.csv master table,477,5,0.0709
7,fndds,input_food,FNDDS ingredient/input-food references,18584,12,5.0229


,dataset,food_count,food_nutrient_rows,nutrient_lookup_key,food_relationship_match_percent,nutrient_relationship_match_percent,category_coverage_percent
0,foundation,395,17008,id,100.0,99.806,100.0
1,fndds,5432,353015,nutrient_nbr,100.0,100.000,100.0


In [17]:
def duplicate_summary(
    dataset,
    table_name,
    df,
    key_columns,
):
    exact_duplicate_rows = int(df.duplicated().sum())

    if (
        df.empty
        or not set(key_columns).issubset(df.columns)
    ):
        duplicate_key_rows = 0
        duplicate_key_groups = 0
    else:
        key_counts = (
            df.groupby(key_columns, dropna=False)
            .size()
            .reset_index(name="row_count")
        )
        duplicate_groups = key_counts[
            key_counts["row_count"] > 1
        ]
        duplicate_key_groups = len(duplicate_groups)
        duplicate_key_rows = int(
            duplicate_groups["row_count"].sum()
        )

    return {
        "dataset": dataset,
        "table": table_name,
        "key_columns": " + ".join(key_columns),
        "exact_duplicate_rows": exact_duplicate_rows,
        "duplicate_key_groups": duplicate_key_groups,
        "rows_in_duplicate_key_groups": duplicate_key_rows,
        "automatic_removal_applied": False,
    }


duplicate_summary_df = pd.DataFrame([
    duplicate_summary(
        "foundation",
        "food",
        foundation_food,
        ["fdc_id"],
    ),
    duplicate_summary(
        "foundation",
        "food_nutrient",
        foundation_food_nutrient,
        ["fdc_id", "nutrient_id"],
    ),
    duplicate_summary(
        "fndds",
        "food",
        fndds_food,
        ["fdc_id"],
    ),
    duplicate_summary(
        "fndds",
        "food_nutrient",
        fndds_food_nutrient,
        ["fdc_id", "nutrient_id"],
    ),
])

foundation_duplicate_nutrient_pairs_df = pd.DataFrame(
    columns=foundation_food_nutrient.columns
)

if (
    not foundation_food_nutrient.empty
    and {"fdc_id", "nutrient_id"}.issubset(
        foundation_food_nutrient.columns
    )
):
    duplicate_mask = (
        foundation_food_nutrient
        .duplicated(
            ["fdc_id", "nutrient_id"],
            keep=False,
        )
    )
    foundation_duplicate_nutrient_pairs_df = (
        foundation_food_nutrient[
            duplicate_mask
        ]
        .sort_values(["fdc_id", "nutrient_id"])
        .copy()
    )

display(duplicate_summary_df)

if not foundation_duplicate_nutrient_pairs_df.empty:
    print(
        "Foundation duplicate nutrient combinations were retained "
        "for manual investigation."
    )
    display(foundation_duplicate_nutrient_pairs_df)

,dataset,table,key_columns,exact_duplicate_rows,duplicate_key_groups,rows_in_duplicate_key_groups,automatic_removal_applied
0,foundation,food,fdc_id,0,0,0,False
1,foundation,food_nutrient,fdc_id + nutrient_id,0,4,8,False
2,fndds,food,fdc_id,0,0,0,False
3,fndds,food_nutrient,fdc_id + nutrient_id,0,0,0,False


Foundation duplicate nutrient combinations were retained for manual investigation.


,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median
160061,35088566,2758990,2066,NaN,NaN,NaN,NaN,NaN,NaN
160062,35088565,2758990,2066,NaN,NaN,NaN,NaN,NaN,NaN
160133,35088574,2758991,2066,NaN,NaN,NaN,NaN,NaN,NaN
160134,35088573,2758991,2066,NaN,NaN,NaN,NaN,NaN,NaN
160208,35088580,2758992,2066,NaN,NaN,NaN,NaN,NaN,NaN
160209,35088579,2758992,2066,NaN,NaN,NaN,NaN,NaN,NaN
169637,35089444,2768188,1106,2.61,NaN,49.0,NaN,NaN,NaN
169640,35089441,2768188,1106,2.61,NaN,49.0,NaN,NaN,NaN


In [18]:
def missing_summary(dataset, table_name, df):
    if df.empty:
        return pd.DataFrame(
            columns=[
                "dataset", "table", "column",
                "missing_count", "missing_percent",
            ]
        )

    return pd.DataFrame({
        "dataset": dataset,
        "table": table_name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": (
            df.isna().mean().values * 100
        ).round(4),
    })


missing_df = pd.concat([
    missing_summary(
        "foundation",
        "food",
        foundation_food,
    ),
    missing_summary(
        "foundation",
        "food_nutrient",
        foundation_food_nutrient,
    ),
    missing_summary(
        "fndds",
        "food",
        fndds_food,
    ),
    missing_summary(
        "fndds",
        "food_nutrient",
        fndds_food_nutrient,
    ),
    missing_summary(
        "fndds",
        "input_food",
        fndds_input_food,
    ),
], ignore_index=True)

display(
    missing_df.sort_values(
        ["dataset", "table", "missing_percent"],
        ascending=[True, True, False],
    )
)

,dataset,table,column,missing_count,missing_percent
14,fndds,food,fdc_id,0,0.0000
15,fndds,food,data_type,0,0.0000
16,fndds,food,description,0,0.0000
17,fndds,food,food_category_id,0,0.0000
18,fndds,food,publication_date,0,0.0000
23,fndds,food_nutrient,data_points,353015,100.0000
24,fndds,food_nutrient,derivation_id,353015,100.0000
25,fndds,food_nutrient,min,353015,100.0000
26,fndds,food_nutrient,max,353015,100.0000
27,fndds,food_nutrient,median,353015,100.0000


## 11. Nutrient completeness

Completeness is calculated against the correct food population for each dataset.
The FNDDS lookup uses its resolved `nutrient_nbr` join identifier.

In [19]:
def nutrient_completeness(
    dataset_name,
    food_df,
    food_nutrient_df,
    nutrient_df,
):
    """
    Calculate nutrient availability across foods.

    Foundation join:
        food_nutrient.nutrient_id -> nutrient.id

    FNDDS join:
        food_nutrient.nutrient_id -> nutrient.nutrient_nbr
    """

    if dataset_name not in {"foundation", "fndds"}:
        raise ValueError(
            "dataset_name must be 'foundation' or 'fndds'"
        )

    observed = food_nutrient_df[
        ["fdc_id", "nutrient_id", "amount"]
    ].copy()

    # Float is required because nutrient_nbr may contain decimal values.
    observed["source_nutrient_id"] = pd.to_numeric(
        observed["nutrient_id"],
        errors="coerce",
    ).astype("float64")

    if dataset_name == "foundation":
        lookup = nutrient_df[
            ["id", "name", "unit_name"]
        ].copy()

        lookup = lookup.rename(columns={
            "id": "source_nutrient_id",
        })

        lookup["nutrient_master_id"] = (
            lookup["source_nutrient_id"]
        )

    else:
        lookup = nutrient_df[
            ["id", "nutrient_nbr", "name", "unit_name"]
        ].copy()

        lookup = lookup.rename(columns={
            "id": "nutrient_master_id",
            "nutrient_nbr": "source_nutrient_id",
        })

    lookup["source_nutrient_id"] = pd.to_numeric(
        lookup["source_nutrient_id"],
        errors="coerce",
    ).astype("float64")

    # Important: remove missing lookup keys before validating the merge.
    lookup = lookup.dropna(
        subset=["source_nutrient_id"]
    ).copy()

    # Audit duplicate valid lookup keys.
    duplicate_lookup_keys = lookup[
        lookup.duplicated(
            subset=["source_nutrient_id"],
            keep=False,
        )
    ].copy()

    if not duplicate_lookup_keys.empty:
        print(
            f"{dataset_name}: "
            f"{duplicate_lookup_keys['source_nutrient_id'].nunique()} "
            "duplicate nutrient lookup keys detected."
        )

        # Keep one deterministic row for each source key.
        lookup = (
            lookup
            .sort_values(
                ["source_nutrient_id", "nutrient_master_id"]
            )
            .drop_duplicates(
                subset=["source_nutrient_id"],
                keep="first",
            )
        )

    merged = observed.merge(
        lookup[
            [
                "source_nutrient_id",
                "nutrient_master_id",
                "name",
                "unit_name",
            ]
        ],
        on="source_nutrient_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    total_foods = food_df["fdc_id"].nunique()

    matched = merged[
        merged["_merge"].eq("both")
        & merged["amount"].notna()
        & merged["name"].notna()
    ].copy()

    completeness_df = (
        matched
        .groupby(
            [
                "source_nutrient_id",
                "nutrient_master_id",
                "name",
                "unit_name",
            ],
            dropna=False,
        )
        .agg(
            foods_with_nutrient=(
                "fdc_id",
                "nunique",
            )
        )
        .reset_index()
    )

    completeness_df["dataset"] = dataset_name
    completeness_df["total_foods"] = total_foods

    completeness_df["completeness_percent"] = (
        completeness_df["foods_with_nutrient"]
        / total_foods
        * 100
    ).round(2)

    completeness_df = completeness_df[
        [
            "dataset",
            "source_nutrient_id",
            "nutrient_master_id",
            "name",
            "unit_name",
            "foods_with_nutrient",
            "total_foods",
            "completeness_percent",
        ]
    ].sort_values(
        "completeness_percent",
        ascending=False,
    ).reset_index(drop=True)

    unmatched_df = (
        merged[
            merged["_merge"].eq("left_only")
        ]
        .groupby(
            "source_nutrient_id",
            dropna=False,
        )
        .agg(
            unmatched_rows=("fdc_id", "size"),
            affected_foods=("fdc_id", "nunique"),
        )
        .reset_index()
        .sort_values(
            "unmatched_rows",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return completeness_df, unmatched_df

In [20]:
(
    foundation_completeness_df,
    foundation_unmatched_nutrients_df,
) = nutrient_completeness(
    "foundation",
    foundation_food,
    foundation_food_nutrient,
    foundation_nutrient,
)

In [21]:
(
    fndds_completeness_df,
    fndds_unmatched_nutrients_df,
) = nutrient_completeness(
    "fndds",
    fndds_food,
    fndds_food_nutrient,
    fndds_nutrient,
)

## 12. Cross-dataset nutrient comparison

In [22]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_unit(value):
    unit = normalize_text(value)
    unit_aliases = {
        "kcal": "kcal",
        "kj": "kj",
        "g": "g",
        "mg": "mg",
        "ug": "ug",
        "µg": "ug",
        "mcg": "ug",
    }
    return unit_aliases.get(unit, unit)


def prepare_comparison_side(completeness_df, prefix):
    required_columns = [
        "source_nutrient_id",
        "nutrient_master_id",
        "name",
        "unit_name",
        "foods_with_nutrient",
        "total_foods",
        "completeness_percent",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in completeness_df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing columns in {prefix} completeness table: "
            f"{missing_columns}"
        )

    result = completeness_df[
        required_columns
    ].copy()

    # Normalized keys allow Foundation and FNDDS nutrient names
    # to be compared safely.
    result["name_key"] = (
        result["name"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    result["unit_key"] = (
        result["unit_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    result = result.rename(columns={
        "source_nutrient_id":
            f"{prefix}_source_nutrient_id",

        "nutrient_master_id":
            f"{prefix}_nutrient_master_id",

        "name":
            f"{prefix}_name",

        "unit_name":
            f"{prefix}_unit_name",

        "foods_with_nutrient":
            f"{prefix}_foods_with_nutrient",

        "total_foods":
            f"{prefix}_total_foods",

        "completeness_percent":
            f"{prefix}_percent",
    })

    return result


foundation_comparison_side = prepare_comparison_side(
    foundation_completeness_df,
    "foundation",
)
fndds_comparison_side = prepare_comparison_side(
    fndds_completeness_df,
    "fndds",
)

nutrient_comparison_df = foundation_comparison_side.merge(
    fndds_comparison_side,
    on=["name_key", "unit_key"],
    how="inner",
)

if not nutrient_comparison_df.empty:
    nutrient_comparison_df["difference_fndds_minus_foundation"] = (
        nutrient_comparison_df["fndds_percent"]
        - nutrient_comparison_df["foundation_percent"]
    ).round(4)

    nutrient_comparison_df["minimum_completeness"] = (
        nutrient_comparison_df[
            ["foundation_percent", "fndds_percent"]
        ].min(axis=1)
    ).round(4)

    nutrient_comparison_df["mean_completeness"] = (
        nutrient_comparison_df[
            ["foundation_percent", "fndds_percent"]
        ].mean(axis=1)
    ).round(4)

    nutrient_comparison_df = (
        nutrient_comparison_df
        .sort_values(
            [
                "minimum_completeness",
                "mean_completeness",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

top_common_nutrients_df = (
    nutrient_comparison_df.head(20).copy()
)

reliable_common_nutrients_df = (
    nutrient_comparison_df[
        nutrient_comparison_df[
            "minimum_completeness"
        ] >= RELIABLE_MIN_COMPLETENESS
    ].copy()
    if not nutrient_comparison_df.empty
    else nutrient_comparison_df.copy()
)

print("Matched common nutrients:", len(nutrient_comparison_df))
print(
    f"Reliable common nutrients "
    f"(minimum completeness >= {RELIABLE_MIN_COMPLETENESS}%):",
    len(reliable_common_nutrients_df),
)
display(top_common_nutrients_df)

Matched common nutrients: 57
Reliable common nutrients (minimum completeness >= 50.0%): 14


,foundation_source_nutrient_id,foundation_nutrient_master_id,foundation_name,foundation_unit_name,foundation_foods_with_nutrient,foundation_total_foods,foundation_percent,name_key,unit_key,fndds_source_nutrient_id,fndds_nutrient_master_id,fndds_name,fndds_unit_name,fndds_foods_with_nutrient,fndds_total_foods,fndds_percent,difference_fndds_minus_foundation,minimum_completeness,mean_completeness
0,1051.0,1051,Water,G,387,395,97.97,water,g,255.0,1051,Water,G,5431,5432,99.98,2.01,97.97,98.975
1,1091.0,1091,"Phosphorus, P",MG,367,395,92.91,"phosphorus, p",mg,305.0,1091,"Phosphorus, P",MG,5431,5432,99.98,7.07,92.91,96.445
2,1090.0,1090,"Magnesium, Mg",MG,367,395,92.91,"magnesium, mg",mg,304.0,1090,"Magnesium, Mg",MG,5431,5432,99.98,7.07,92.91,96.445
3,1089.0,1089,"Iron, Fe",MG,367,395,92.91,"iron, fe",mg,303.0,1089,"Iron, Fe",MG,5431,5432,99.98,7.07,92.91,96.445
4,1087.0,1087,"Calcium, Ca",MG,367,395,92.91,"calcium, ca",mg,301.0,1087,"Calcium, Ca",MG,5431,5432,99.98,7.07,92.91,96.445
5,1098.0,1098,"Copper, Cu",MG,367,395,92.91,"copper, cu",mg,312.0,1098,"Copper, Cu",MG,5431,5432,99.98,7.07,92.91,96.445
6,1095.0,1095,"Zinc, Zn",MG,367,395,92.91,"zinc, zn",mg,309.0,1095,"Zinc, Zn",MG,5431,5432,99.98,7.07,92.91,96.445
7,1092.0,1092,"Potassium, K",MG,367,395,92.91,"potassium, k",mg,306.0,1092,"Potassium, K",MG,5431,5432,99.98,7.07,92.91,96.445
8,1003.0,1003,Protein,G,353,395,89.37,protein,g,203.0,1003,Protein,G,5431,5432,99.98,10.61,89.37,94.675
9,1093.0,1093,"Sodium, Na",MG,349,395,88.35,"sodium, na",mg,307.0,1093,"Sodium, Na",MG,5431,5432,99.98,11.63,88.35,94.165


## 13. Dataset-specific selected nutrient map

Both the nutrient master identifier and the dataset-specific source join value are retained.
For FNDDS the source join value should contain values such as 208, 203, 205, and 204,
rather than the master IDs 1008, 1003, 1005, and 1004.

In [23]:
TARGET_NUTRIENTS = {
    "energy_kcal": {
        "aliases": [
            "Energy",
            "Energy (Atwater General Factors)",
            "Energy (Atwater Specific Factors)",
        ],
        "unit": "kcal",
    },
    "protein_g": {
        "aliases": ["Protein"],
        "unit": "g",
    },
    "carbohydrate_g": {
        "aliases": ["Carbohydrate, by difference"],
        "unit": "g",
    },
    "fat_g": {
        "aliases": ["Total lipid (fat)"],
        "unit": "g",
    },
    "fiber_g": {
        "aliases": [
            "Fiber, total dietary",
            "Fiber, total",
        ],
        "unit": "g",
    },
    "sugar_g": {
        "aliases": [
            "Sugars, total including NLEA",
            "Sugars, total",
            "Total Sugars",
        ],
        "unit": "g",
    },
    "sodium_mg": {
        "aliases": ["Sodium, Na"],
        "unit": "mg",
    },
}


def create_nutrient_map(
    dataset,
    nutrient_df,
    nutrient_lookup_key,
    food_nutrient_df,
):
    rows = []

    if nutrient_df.empty:
        nutrient_df = pd.DataFrame(
            columns=[
                "id", "nutrient_nbr",
                "name", "unit_name",
            ]
        )

    lookup = nutrient_df.copy()
    lookup["name_key"] = lookup.get(
        "name",
        pd.Series(index=lookup.index, dtype=object),
    ).map(normalize_text)
    lookup["unit_key"] = lookup.get(
        "unit_name",
        pd.Series(index=lookup.index, dtype=object),
    ).map(normalize_unit)

    observed_source_ids = set(
        pd.to_numeric(
            food_nutrient_df.get(
                "nutrient_id",
                pd.Series(dtype=float),
            ),
            errors="coerce",
        ).dropna().tolist()
    )

    for normalized_field, specification in TARGET_NUTRIENTS.items():
        alias_keys = {
            normalize_text(alias)
            for alias in specification["aliases"]
        }
        unit_key = normalize_unit(specification["unit"])

        matches = lookup[
            lookup["name_key"].isin(alias_keys)
            & lookup["unit_key"].eq(unit_key)
        ].copy()

        if matches.empty:
            rows.append({
                "dataset": dataset,
                "normalized_field": normalized_field,
                "nutrient_master_id": pd.NA,
                "nutrient_nbr": pd.NA,
                "source_join_key": nutrient_lookup_key,
                "source_join_value": pd.NA,
                "original_name": None,
                "unit_name": specification["unit"],
                "observed_in_food_nutrient": False,
                "status": "not_found_in_nutrient_master",
            })
            continue

        selected = matches.sort_values(
            ["id", "nutrient_nbr"],
            na_position="last",
        ).iloc[0]

        source_join_value = selected.get(
            nutrient_lookup_key,
            pd.NA,
        )
        numeric_source_join_value = pd.to_numeric(
            pd.Series([source_join_value]),
            errors="coerce",
        ).iloc[0]

        observed = (
            not pd.isna(numeric_source_join_value)
            and numeric_source_join_value
            in observed_source_ids
        )

        rows.append({
            "dataset": dataset,
            "normalized_field": normalized_field,
            "nutrient_master_id": selected.get("id", pd.NA),
            "nutrient_nbr": selected.get(
                "nutrient_nbr",
                pd.NA,
            ),
            "source_join_key": nutrient_lookup_key,
            "source_join_value": source_join_value,
            "original_name": selected.get("name"),
            "unit_name": selected.get("unit_name"),
            "observed_in_food_nutrient": observed,
            "status": (
                "available_and_observed"
                if observed
                else "available_but_not_observed"
            ),
        })

    result = pd.DataFrame(rows)

    for column in [
        "nutrient_master_id",
        "nutrient_nbr",
        "source_join_value",
    ]:
        if column in result.columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="coerce",
            ).astype("Int64")

    return result


def _coerce_integer_like(series):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.empty:
        return numeric.astype("Int64")
    non_null = numeric.dropna()
    if non_null.empty:
        return numeric.astype("Int64")
    if ((non_null % 1) == 0).all():
        return numeric.astype("Int64")
    return numeric.astype("Float64")


def create_nutrient_map(
    dataset,
    nutrient_df,
    nutrient_lookup_key,
    food_nutrient_df,
):
    rows = []

    if nutrient_df.empty:
        nutrient_df = pd.DataFrame(
            columns=[
                "id", "nutrient_nbr",
                "name", "unit_name",
            ]
        )

    lookup = nutrient_df.copy()
    lookup["name_key"] = lookup.get(
        "name",
        pd.Series(index=lookup.index, dtype=object),
    ).map(normalize_text)
    lookup["unit_key"] = lookup.get(
        "unit_name",
        pd.Series(index=lookup.index, dtype=object),
    ).map(normalize_unit)

    observed_source_ids = set(
        pd.to_numeric(
            food_nutrient_df.get(
                "nutrient_id",
                pd.Series(dtype=float),
            ),
            errors="coerce",
        ).dropna().tolist()
    )

    for normalized_field, specification in TARGET_NUTRIENTS.items():
        alias_keys = {
            normalize_text(alias)
            for alias in specification["aliases"]
        }
        unit_key = normalize_unit(specification["unit"])

        matches = lookup[
            lookup["name_key"].isin(alias_keys)
            & lookup["unit_key"].eq(unit_key)
        ].copy()

        if matches.empty:
            rows.append({
                "dataset": dataset,
                "normalized_field": normalized_field,
                "nutrient_master_id": pd.NA,
                "nutrient_nbr": pd.NA,
                "source_join_key": nutrient_lookup_key,
                "source_join_value": pd.NA,
                "original_name": None,
                "unit_name": specification["unit"],
                "observed_in_food_nutrient": False,
                "status": "not_found_in_nutrient_master",
            })
            continue

        selected = matches.sort_values(
            ["id", "nutrient_nbr"],
            na_position="last",
        ).iloc[0]

        source_join_value = selected.get(
            nutrient_lookup_key,
            pd.NA,
        )
        numeric_source_join_value = pd.to_numeric(
            pd.Series([source_join_value]),
            errors="coerce",
        ).iloc[0]

        observed = (
            not pd.isna(numeric_source_join_value)
            and numeric_source_join_value
            in observed_source_ids
        )

        rows.append({
            "dataset": dataset,
            "normalized_field": normalized_field,
            "nutrient_master_id": selected.get("id", pd.NA),
            "nutrient_nbr": selected.get(
                "nutrient_nbr",
                pd.NA,
            ),
            "source_join_key": nutrient_lookup_key,
            "source_join_value": source_join_value,
            "original_name": selected.get("name"),
            "unit_name": selected.get("unit_name"),
            "observed_in_food_nutrient": observed,
            "status": (
                "available_and_observed"
                if observed
                else "available_but_not_observed"
            ),
        })

    result = pd.DataFrame(rows)

    for column in [
        "nutrient_master_id",
        "nutrient_nbr",
        "source_join_value",
    ]:
        if column in result.columns:
            result[column] = _coerce_integer_like(result[column])

    return result


foundation_nutrient_map = create_nutrient_map(
    "foundation",
    foundation_nutrient,
    FOUNDATION_NUTRIENT_KEY,
    foundation_food_nutrient,
)

fndds_nutrient_map = create_nutrient_map(
    "fndds",
    fndds_nutrient,
    FNDDS_NUTRIENT_KEY,
    fndds_food_nutrient,
)

selected_nutrient_map_df = pd.concat(
    [
        foundation_nutrient_map,
        fndds_nutrient_map,
    ],
    ignore_index=True,
)

display(selected_nutrient_map_df)

,dataset,normalized_field,nutrient_master_id,nutrient_nbr,source_join_key,source_join_value,original_name,unit_name,observed_in_food_nutrient,status
0,foundation,energy_kcal,1008,208.0,id,1008.0,Energy,KCAL,True,available_and_observed
1,foundation,protein_g,1003,203.0,id,1003.0,Protein,G,True,available_and_observed
2,foundation,carbohydrate_g,1005,205.0,id,1005.0,"Carbohydrate, by difference",G,True,available_and_observed
3,foundation,fat_g,1004,204.0,id,1004.0,Total lipid (fat),G,True,available_and_observed
4,foundation,fiber_g,1079,291.0,id,1079.0,"Fiber, total dietary",G,True,available_and_observed
5,foundation,sugar_g,1063,269.3,id,1063.0,"Sugars, Total",G,True,available_and_observed
6,foundation,sodium_mg,1093,307.0,id,1093.0,"Sodium, Na",MG,True,available_and_observed
7,fndds,energy_kcal,1008,208.0,nutrient_nbr,208.0,Energy,KCAL,True,available_and_observed
8,fndds,protein_g,1003,203.0,nutrient_nbr,203.0,Protein,G,True,available_and_observed
9,fndds,carbohydrate_g,1005,205.0,nutrient_nbr,205.0,"Carbohydrate, by difference",G,True,available_and_observed


## 14. Dataset interpretation and CeylonSavour field mapping

In [24]:
def observed_nutrient_count(food_nutrient_df):
    if (
        food_nutrient_df.empty
        or "nutrient_id" not in food_nutrient_df.columns
    ):
        return 0
    return int(
        food_nutrient_df[
            "nutrient_id"
        ].dropna().nunique()
    )


def matched_observed_nutrient_count(completeness_df):
    if completeness_df.empty:
        return 0
    return int(
        completeness_df[
            "source_nutrient_id"
        ].dropna().nunique()
    )


dataset_comparison_df = pd.DataFrame([
    {
        "dataset": "foundation",
        "dataset_label": "Foundation Foods",
        "food_count": (
            foundation_food["fdc_id"].dropna().nunique()
            if "fdc_id" in foundation_food.columns
            else 0
        ),
        "food_nutrient_rows": len(foundation_food_nutrient),
        "nutrient_master_count": len(foundation_nutrient),
        "observed_nutrient_count": (
            observed_nutrient_count(
                foundation_food_nutrient
            )
        ),
        "matched_observed_nutrient_count": (
            matched_observed_nutrient_count(
                foundation_completeness_df
            )
        ),
        "nutrient_lookup_key": FOUNDATION_NUTRIENT_KEY,
        "category_coverage_percent": (
            foundation_category_coverage
        ),
        "ingredient_reference_coverage_percent": 0.0,
        "best_use": (
            "Basic foods, ingredients, and nutrient "
            "reference values"
        ),
    },
    {
        "dataset": "fndds",
        "dataset_label": "FNDDS",
        "food_count": (
            fndds_food["fdc_id"].dropna().nunique()
            if "fdc_id" in fndds_food.columns
            else 0
        ),
        "food_nutrient_rows": len(fndds_food_nutrient),
        "nutrient_master_count": len(fndds_nutrient),
        "observed_nutrient_count": (
            observed_nutrient_count(
                fndds_food_nutrient
            )
        ),
        "matched_observed_nutrient_count": (
            matched_observed_nutrient_count(
                fndds_completeness_df
            )
        ),
        "nutrient_lookup_key": FNDDS_NUTRIENT_KEY,
        "category_coverage_percent": (
            fndds_category_coverage
        ),
        "ingredient_reference_coverage_percent": (
            input_food_coverage_percent
        ),
        "best_use": (
            "Prepared foods, commonly consumed foods, "
            "WWEIA categories, and ingredient references"
        ),
    },
])

display(dataset_comparison_df)

,dataset,dataset_label,food_count,food_nutrient_rows,nutrient_master_count,observed_nutrient_count,matched_observed_nutrient_count,nutrient_lookup_key,category_coverage_percent,ingredient_reference_coverage_percent,best_use
0,foundation,Foundation Foods,395,17008,477,235,234,id,100.0,0.0000,"Basic foods, ingredients, and nutrient reference values"
1,fndds,FNDDS,5432,353015,477,65,65,nutrient_nbr,100.0,99.9816,"Prepared foods, commonly consumed foods, WWEIA categories, and ingredient references"


In [25]:
ceylonsavour_mapping_df = pd.DataFrame([
    {
        "ceylonsavour_field": "dish_name",
        "foundation_source": "selected food.description",
        "fndds_source": "food.description",
        "interpretation": "Direct reference",
    },
    {
        "ceylonsavour_field": "source_food_id",
        "foundation_source": "selected food.fdc_id",
        "fndds_source": "food.fdc_id",
        "interpretation": "Direct reference",
    },
    {
        "ceylonsavour_field": "food_category",
        "foundation_source": "food_category.description",
        "fndds_source": (
            "wweia_food_category_description"
        ),
        "interpretation": (
            "Use the dataset-specific category relationship"
        ),
    },
    {
        "ceylonsavour_field": "nutrient",
        "foundation_source": (
            "food_nutrient.nutrient_id -> nutrient.id"
        ),
        "fndds_source": (
            "food_nutrient.nutrient_id -> "
            "nutrient.nutrient_nbr"
        ),
        "interpretation": (
            "Store master ID, nutrient number, amount, and unit"
        ),
    },
    {
        "ceylonsavour_field": "portion_grams",
        "foundation_source": "food_portion.gram_weight",
        "fndds_source": "food_portion.gram_weight",
        "interpretation": "Optional portion reference",
    },
    {
        "ceylonsavour_field": "ingredients",
        "foundation_source": (
            "Not consistently available for final foods"
        ),
        "fndds_source": (
            "input_food.csv provides ingredient/input-food "
            "references for many survey foods"
        ),
        "interpretation": (
            "Useful for international structural study, but "
            "not complete or authoritative for Sri Lankan dishes"
        ),
    },
    {
        "ceylonsavour_field": "taste_profile",
        "foundation_source": "Not available",
        "fndds_source": "Not available",
        "interpretation": (
            "Create from FlavorDB, surveys, and expert annotation"
        ),
    },
    {
        "ceylonsavour_field": "dietary_tags",
        "foundation_source": "Not directly available",
        "fndds_source": "Can be partially derived from input foods",
        "interpretation": (
            "Derive and validate using Sri Lankan ingredients "
            "and preparation methods"
        ),
    },
    {
        "ceylonsavour_field": "allergy_tags",
        "foundation_source": "Not directly available",
        "fndds_source": "Can be partially derived from input foods",
        "interpretation": (
            "Derive with explicit allergen rules and manual validation"
        ),
    },
    {
        "ceylonsavour_field": "authenticity_and_region",
        "foundation_source": "Not available",
        "fndds_source": "Not applicable to Sri Lankan authenticity",
        "interpretation": (
            "Collect from Sri Lankan experts and cultural sources"
        ),
    },
    {
        "ceylonsavour_field": "tourist_friendliness",
        "foundation_source": "Not available",
        "fndds_source": "Not available",
        "interpretation": (
            "Create from tourist surveys and expert-defined rules"
        ),
    },
])

display(ceylonsavour_mapping_df)

,ceylonsavour_field,foundation_source,fndds_source,interpretation
0,dish_name,selected food.description,food.description,Direct reference
1,source_food_id,selected food.fdc_id,food.fdc_id,Direct reference
2,food_category,food_category.description,wweia_food_category_description,Use the dataset-specific category relationship
3,nutrient,food_nutrient.nutrient_id -> nutrient.id,food_nutrient.nutrient_id -> nutrient.nutrient_nbr,"Store master ID, nutrient number, amount, and unit"
4,portion_grams,food_portion.gram_weight,food_portion.gram_weight,Optional portion reference
5,ingredients,Not consistently available for final foods,input_food.csv provides ingredient/input-food references for many survey foods,"Useful for international structural study, but not complete or authoritative for Sri Lankan dishes"
6,taste_profile,Not available,Not available,"Create from FlavorDB, surveys, and expert annotation"
7,dietary_tags,Not directly available,Can be partially derived from input foods,Derive and validate using Sri Lankan ingredients and preparation methods
8,allergy_tags,Not directly available,Can be partially derived from input foods,Derive with explicit allergen rules and manual validation
9,authenticity_and_region,Not available,Not applicable to Sri Lankan authenticity,Collect from Sri Lankan experts and cultural sources


## 15. Normalized USDA food reference

Duplicate `(fdc_id, nutrient_id)` observations are retained in the audit output.
When a normalized target field needs one value, the documented aggregation rule
is `mean`; the original rows are never deleted automatically.

In [26]:
def create_reference_dataset(
    food_base_df,
    food_nutrient_df,
    nutrient_map_df,
    source_dataset,
):
    nutrient_columns = list(TARGET_NUTRIENTS.keys())

    result = food_base_df.copy()

    available_map = nutrient_map_df[
        nutrient_map_df["observed_in_food_nutrient"].eq(True)
        & nutrient_map_df["source_join_value"].notna()
    ][
        ["source_join_value", "normalized_field"]
    ].drop_duplicates("source_join_value")

    if (
        not result.empty
        and not food_nutrient_df.empty
        and not available_map.empty
        and {"fdc_id", "nutrient_id", "amount"}.issubset(
            food_nutrient_df.columns
        )
    ):
        nutrient_values = (
            food_nutrient_df[
                ["fdc_id", "nutrient_id", "amount"]
            ]
            .merge(
                available_map.rename(columns={
                    "source_join_value": "nutrient_id",
                }),
                on="nutrient_id",
                how="inner",
            )
            .pivot_table(
                index="fdc_id",
                columns="normalized_field",
                values="amount",
                aggfunc=NORMALIZED_DUPLICATE_AGGREGATION,
            )
            .reset_index()
        )

        result = result.merge(
            nutrient_values,
            on="fdc_id",
            how="left",
        )

    for column in nutrient_columns:
        if column not in result.columns:
            result[column] = np.nan

    result["source_dataset"] = source_dataset
    result["available_nutrient_count"] = (
        result[nutrient_columns]
        .notna()
        .sum(axis=1)
    )

    ordered_columns = [
        "source_dataset", "fdc_id",
        "food_name", "food_category",
        *nutrient_columns,
        "available_nutrient_count",
    ]

    for column in ordered_columns:
        if column not in result.columns:
            result[column] = pd.NA

    return result[ordered_columns]


foundation_base = pd.DataFrame(
    columns=["fdc_id", "food_name", "food_category"]
)

if not foundation_food_with_category.empty:
    foundation_base = (
        foundation_food_with_category[
            ["fdc_id", "description", "food_category"]
        ]
        .rename(columns={"description": "food_name"})
        .drop_duplicates("fdc_id")
    )


fndds_base = pd.DataFrame(
    columns=["fdc_id", "food_name", "food_category"]
)

if not fndds_food_with_category.empty:
    if (
        "wweia_food_category_description"
        in fndds_food_with_category.columns
    ):
        fndds_base = (
            fndds_food_with_category[
                [
                    "fdc_id",
                    "description",
                    "wweia_food_category_description",
                ]
            ]
            .rename(columns={
                "description": "food_name",
                "wweia_food_category_description":
                    "food_category",
            })
            .drop_duplicates("fdc_id")
        )
    else:
        fndds_base = (
            fndds_food_with_category[
                ["fdc_id", "description"]
            ]
            .rename(columns={"description": "food_name"})
            .assign(food_category=pd.NA)
            .drop_duplicates("fdc_id")
        )


foundation_reference_df = create_reference_dataset(
    foundation_base,
    foundation_food_nutrient,
    foundation_nutrient_map,
    "foundation",
)

fndds_reference_df = create_reference_dataset(
    fndds_base,
    fndds_food_nutrient,
    fndds_nutrient_map,
    "fndds",
)

usda_reference_df = pd.concat(
    [
        foundation_reference_df,
        fndds_reference_df,
    ],
    ignore_index=True,
)

usda_reference_df = (
    usda_reference_df
    .sort_values(
        ["source_dataset", "food_name"],
        na_position="last",
    )
    .reset_index(drop=True)
)

usda_reference_quality_df = (
    usda_reference_df[
        usda_reference_df[
            "available_nutrient_count"
        ] >= QUALITY_MIN_SELECTED_NUTRIENTS
    ].copy()
)

if usda_reference_df.empty:
    reference_summary_df = pd.DataFrame(
        columns=[
            "source_dataset", "food_count",
            "category_count",
            "average_available_nutrients",
            "quality_food_count",
        ]
    )
else:
    reference_summary_df = (
        usda_reference_df
        .groupby("source_dataset", dropna=False)
        .agg(
            food_count=("fdc_id", "nunique"),
            category_count=("food_category", "nunique"),
            average_available_nutrients=(
                "available_nutrient_count",
                "mean",
            ),
        )
        .reset_index()
    )
    quality_counts = (
        usda_reference_quality_df
        .groupby("source_dataset")["fdc_id"]
        .nunique()
        .rename("quality_food_count")
        .reset_index()
    )
    reference_summary_df = (
        reference_summary_df
        .merge(
            quality_counts,
            on="source_dataset",
            how="left",
        )
    )
    reference_summary_df["quality_food_count"] = (
        reference_summary_df[
            "quality_food_count"
        ].fillna(0).astype(int)
    )
    reference_summary_df[
        "average_available_nutrients"
    ] = reference_summary_df[
        "average_available_nutrients"
    ].round(4)

display(reference_summary_df)

,source_dataset,food_count,category_count,average_available_nutrients,quality_food_count
0,fndds,5432,172,5.9989,5431
1,foundation,395,19,4.5747,332


## 16. Final validation

In [27]:
def relationship_value(dataset, relationship, column):
    match = relationship_validation_df[
        relationship_validation_df["dataset"].eq(dataset)
        & relationship_validation_df[
            "relationship"
        ].eq(relationship)
    ]
    if match.empty:
        return 0
    return match.iloc[0][column]


fndds_target_observed_count = int(
    fndds_nutrient_map[
        "observed_in_food_nutrient"
    ].sum()
) if not fndds_nutrient_map.empty else 0

fndds_normalized_value_count = int(
    fndds_reference_df[
        list(TARGET_NUTRIENTS.keys())
    ].notna().sum().sum()
) if not fndds_reference_df.empty else 0

validation_checks_df = pd.DataFrame([
    {
        "check": "Both USDA releases ready",
        "value": USDA_READY,
        "expected": True,
        "passed": USDA_READY,
    },
    {
        "check": "Actual Foundation Foods selected",
        "value": len(foundation_food),
        "expected": "> 0",
        "passed": len(foundation_food) > 0,
    },
    {
        "check": "Foundation selection excludes generic extra records",
        "value": (
            len(foundation_food_all) - len(foundation_food)
        ),
        "expected": ">= 0; normally > 0",
        "passed": (
            len(foundation_food_all) >= len(foundation_food)
        ),
    },
    {
        "check": "Foundation nutrient value relationship",
        "value": relationship_value(
            "foundation",
            "food_nutrient to nutrient",
            "match_percent",
        ),
        "expected": f">= {RELATIONSHIP_VALID_THRESHOLD}%",
        "passed": relationship_value(
            "foundation",
            "food_nutrient to nutrient",
            "match_percent",
        ) >= RELATIONSHIP_VALID_THRESHOLD,
    },
    {
        "check": "FNDDS nutrient lookup key",
        "value": FNDDS_NUTRIENT_KEY,
        "expected": "nutrient_nbr",
        "passed": FNDDS_NUTRIENT_KEY == "nutrient_nbr",
    },
    {
        "check": "FNDDS nutrient value relationship",
        "value": relationship_value(
            "fndds",
            "food_nutrient to nutrient",
            "match_percent",
        ),
        "expected": f">= {RELATIONSHIP_VALID_THRESHOLD}%",
        "passed": relationship_value(
            "fndds",
            "food_nutrient to nutrient",
            "match_percent",
        ) >= RELATIONSHIP_VALID_THRESHOLD,
    },
    {
        "check": "FNDDS completeness names populated",
        "value": int(
            fndds_completeness_df["name"].isna().sum()
        ) if "name" in fndds_completeness_df.columns else 0,
        "expected": 0,
        "passed": (
            int(
                fndds_completeness_df[
                    "name"
                ].isna().sum()
            ) == 0
            if "name" in fndds_completeness_df.columns
            else False
        ),
    },
    {
        "check": "Common nutrient comparison populated",
        "value": len(nutrient_comparison_df),
        "expected": "> 0",
        "passed": len(nutrient_comparison_df) > 0,
    },
    {
        "check": "FNDDS selected nutrients use observed join values",
        "value": fndds_target_observed_count,
        "expected": "> 0",
        "passed": fndds_target_observed_count > 0,
    },
    {
        "check": "FNDDS normalized nutrient values created",
        "value": fndds_normalized_value_count,
        "expected": "> 0",
        "passed": fndds_normalized_value_count > 0,
    },
    {
        "check": "Quality normalized reference created",
        "value": len(usda_reference_quality_df),
        "expected": "> 0",
        "passed": len(usda_reference_quality_df) > 0,
    },
])

display(validation_checks_df)

if validation_checks_df["passed"].all():
    print(
        "All USDA validation checks passed. "
        "The notebook is ready to proceed to FlavorDB."
    )
else:
    print(
        "One or more checks did not pass. "
        "Use the value and expected columns to identify the issue."
    )

,check,value,expected,passed
0,Both USDA releases ready,True,True,True
1,Actual Foundation Foods selected,395,> 0,True
2,Foundation selection excludes generic extra records,87595,>= 0; normally > 0,True
3,Foundation nutrient value relationship,99.806,>= 95.0%,True
4,FNDDS nutrient lookup key,nutrient_nbr,nutrient_nbr,True
5,FNDDS nutrient value relationship,100.0,>= 95.0%,True
6,FNDDS completeness names populated,0,0,True
7,Common nutrient comparison populated,57,> 0,True
8,FNDDS selected nutrients use observed join values,6,> 0,True
9,FNDDS normalized nutrient values created,32586,> 0,True


All USDA validation checks passed. The notebook is ready to proceed to FlavorDB.


## 17. Final findings

In [28]:
final_findings_df = pd.DataFrame([
    {
        "area": "Foundation food population",
        "finding": (
            "Foundation completeness and normalized outputs use only "
            "actual Foundation Food records selected through "
            "foundation_food.csv, with a data_type fallback."
        ),
    },
    {
        "area": "FNDDS nutrient identifier",
        "finding": (
            "The FNDDS food_nutrient.nutrient_id values are resolved "
            "against nutrient.nutrient_nbr. The nutrient master ID and "
            "source join value are both retained."
        ),
    },
    {
        "area": "Relationship validation",
        "finding": (
            "Relationships are validated using matched and unmatched "
            "values, null keys, duplicate parent keys, and a threshold; "
            "column existence alone is not treated as valid."
        ),
    },
    {
        "area": "Observed nutrient counts",
        "finding": (
            "Dataset interpretation separates nutrient master-table "
            "size from nutrients actually observed in food_nutrient."
        ),
    },
    {
        "area": "Duplicate nutrient combinations",
        "finding": (
            "Duplicate Foundation fdc_id + nutrient_id combinations are "
            "retained for manual inspection. Normalized target values use "
            f"the documented {NORMALIZED_DUPLICATE_AGGREGATION} aggregation."
        ),
    },
    {
        "area": "FNDDS ingredients",
        "finding": (
            "FNDDS input_food.csv is included as an international "
            "ingredient-reference source, but it is not a complete or "
            "authoritative ingredient source for Sri Lankan dishes."
        ),
    },
    {
        "area": "CeylonSavour use",
        "finding": (
            "USDA supports nutrient standards, portions, categories, and "
            "international reference matching. Sri Lankan taste profiles, "
            "authenticity, regional relevance, dietary/allergen validation, "
            "and tourist friendliness require separate collection."
        ),
    },
])

display(final_findings_df)

,area,finding
0,Foundation food population,"Foundation completeness and normalized outputs use only actual Foundation Food records selected through foundation_food.csv, with a data_type fallback."
1,FNDDS nutrient identifier,The FNDDS food_nutrient.nutrient_id values are resolved against nutrient.nutrient_nbr. The nutrient master ID and source join value are both retained.
2,Relationship validation,"Relationships are validated using matched and unmatched values, null keys, duplicate parent keys, and a threshold; column existence alone is not treated as valid."
3,Observed nutrient counts,Dataset interpretation separates nutrient master-table size from nutrients actually observed in food_nutrient.
4,Duplicate nutrient combinations,Duplicate Foundation fdc_id + nutrient_id combinations are retained for manual inspection. Normalized target values use the documented mean aggregation.
5,FNDDS ingredients,"FNDDS input_food.csv is included as an international ingredient-reference source, but it is not a complete or authoritative ingredient source for Sri Lankan dishes."
6,CeylonSavour use,"USDA supports nutrient standards, portions, categories, and international reference matching. Sri Lankan taste profiles, authenticity, regional relevance, dietary/allergen vali..."


## 18. Export all USDA outputs and create an exact manifest

Previous `usda_*.csv` and `usda_*.json` files in the processed USDA directory
are removed before export so stale contradictory outputs cannot remain.

In [29]:
if CLEAN_PREVIOUS_USDA_OUTPUTS:
    for pattern in ["usda_*.csv", "usda_*.json"]:
        for old_path in PROCESSED_USDA_ROOT.glob(pattern):
            try:
                old_path.unlink()
            except Exception as error:
                warnings.warn(
                    f"Could not remove old output {old_path}: {error}"
                )

# Create a fallback dataframe for unmatched nutrient identifiers
# when no dedicated matching output was created earlier.
if "unmatched_nutrient_identifiers_df" not in globals():
    if "resolution_rows" in globals() and resolution_rows:
        unmatched_nutrient_identifiers_df = pd.DataFrame(
            [
                {
                    "dataset": row["dataset"],
                    "candidate_nutrient_key": row["candidate_nutrient_key"],
                    "non_null_rows": row["non_null_rows"],
                    "matched_rows": row["matched_rows"],
                    "unmatched_rows": row["unmatched_rows"],
                    "match_percent": row["match_percent"],
                }
                for row in resolution_rows
                if row["unmatched_rows"] > 0
            ]
        )
    else:
        unmatched_nutrient_identifiers_df = pd.DataFrame(
            columns=[
                "dataset",
                "candidate_nutrient_key",
                "non_null_rows",
                "matched_rows",
                "unmatched_rows",
                "match_percent",
            ]
        )


EXPORT_TABLES = {
    "usda_release_status.csv": release_status_df,
    "usda_file_inventory.csv": file_inventory_df,
    "usda_inventory_summary.csv": inventory_summary_df,
    "usda_csv_schema_summary.csv": schema_file_summary_df,
    "usda_csv_schema_long.csv": schema_long_df,
    "usda_table_role_mapping.csv": table_role_mapping_df,
    "usda_relationship_schema_validation.csv":
        relationship_schema_validation_df,
    "usda_relationship_validation.csv":
        relationship_validation_df,
    "usda_relationship_coverage.csv":
        relationship_coverage_df,
    "usda_foundation_record_type_summary.csv":
        foundation_record_type_summary_df,
    "usda_nutrient_key_resolution.csv":
        nutrient_key_resolution_df,
    "usda_core_table_summary.csv":
        table_summary_df,
    "usda_core_join_summary.csv":
        core_join_summary_df,
    "usda_duplicate_summary.csv":
        duplicate_summary_df,
    "usda_foundation_duplicate_nutrient_pairs.csv":
        foundation_duplicate_nutrient_pairs_df,
    "usda_category_strategy.csv":
        category_strategy_df,
    "usda_category_coverage.csv":
        category_coverage_df,
    "usda_fndds_ingredient_reference_summary.csv":
        fndds_ingredient_reference_summary_df,
    "usda_missing_values.csv":
        missing_df,
    "usda_foundation_nutrient_completeness.csv":
        foundation_completeness_df,
    "usda_fndds_nutrient_completeness.csv":
        fndds_completeness_df,
    "usda_nutrient_completeness.csv":
        nutrient_completeness,
    "usda_unmatched_nutrient_identifiers.csv":
        unmatched_nutrient_identifiers_df,
    "usda_nutrient_comparison.csv":
        nutrient_comparison_df,
    "usda_top_common_nutrients.csv":
        top_common_nutrients_df,
    "usda_reliable_common_nutrients.csv":
        reliable_common_nutrients_df,
    "usda_selected_nutrient_map.csv":
        selected_nutrient_map_df,
    "usda_dataset_interpretation.csv":
        dataset_comparison_df,
    "usda_ceylonsavour_field_mapping.csv":
        ceylonsavour_mapping_df,
    "usda_normalized_food_reference.csv":
        usda_reference_df,
    "usda_normalized_food_reference_quality.csv":
        usda_reference_quality_df,
    "usda_normalized_reference_summary.csv":
        reference_summary_df,
    "usda_validation_checks.csv":
        validation_checks_df,
    "usda_final_findings.csv":
        final_findings_df,
}


export_rows = []
generated_files = []

for filename, dataframe in EXPORT_TABLES.items():
    output_path = PROCESSED_USDA_ROOT / filename

    try:
        dataframe.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig",
        )
        status = "saved"
        generated_files.append(filename)
    except Exception as error:
        status = f"not_saved: {error}"

    try:
        row_count = len(dataframe)
    except TypeError:
        row_count = 0

    export_rows.append({
        "filename": filename,
        "rows": row_count,
        "status": status,
        "path": str(output_path),
    })


schema_json_filename = "usda_csv_schema_inventory.json"
schema_json_path = (
    PROCESSED_USDA_ROOT / schema_json_filename
)

try:
    schema_json_records = (
        schema_file_summary_df
        .assign(
            columns=schema_file_summary_df[
                "columns"
            ].apply(
                lambda value: (
                    value
                    if isinstance(value, list)
                    else []
                )
            )
        )
        .to_dict(orient="records")
    )

    with open(
        schema_json_path,
        "w",
        encoding="utf-8",
    ) as schema_json_file:
        json.dump(
            schema_json_records,
            schema_json_file,
            indent=2,
            ensure_ascii=False,
        )

    generated_files.append(schema_json_filename)
    export_rows.append({
        "filename": schema_json_filename,
        "rows": len(schema_json_records),
        "status": "saved",
        "path": str(schema_json_path),
    })
except Exception as error:
    export_rows.append({
        "filename": schema_json_filename,
        "rows": 0,
        "status": f"not_saved: {error}",
        "path": str(schema_json_path),
    })


export_summary_df = pd.DataFrame(export_rows)

export_summary_filename = "usda_export_summary.csv"
export_summary_path = (
    PROCESSED_USDA_ROOT / export_summary_filename
)
export_summary_df.to_csv(
    export_summary_path,
    index=False,
    encoding="utf-8-sig",
)
generated_files.append(export_summary_filename)


manifest_filename = "usda_analysis_manifest.json"
manifest_path = (
    PROCESSED_USDA_ROOT / manifest_filename
)

manifest = {
    "generated_at_utc": RUN_TIMESTAMP_UTC,
    "project_root": str(PROJECT_ROOT),
    "project_root_discovery_method": PROJECT_ROOT_METHOD,
    "usda_raw_root": str(USDA_ROOT),
    "processed_output_root": str(PROCESSED_USDA_ROOT),
    "foundation_release": (
        str(FOUNDATION_RELEASE)
        if FOUNDATION_RELEASE is not None
        else None
    ),
    "fndds_release": (
        str(FNDDS_RELEASE)
        if FNDDS_RELEASE is not None
        else None
    ),
    "usda_ready": USDA_READY,
    "schema_inventory_scope": (
        "CSV files under extracted directories only; "
        "archive/checksum CSV files are excluded"
    ),
    "foundation_filter_method": FOUNDATION_FILTER_METHOD,
    "foundation_generic_food_rows": int(
        len(foundation_food_all)
    ),
    "foundation_final_food_rows": int(
        len(foundation_food)
    ),
    "foundation_nutrient_lookup_key":
        FOUNDATION_NUTRIENT_KEY,
    "fndds_nutrient_lookup_key":
        FNDDS_NUTRIENT_KEY,
    "relationship_valid_threshold_percent":
        RELATIONSHIP_VALID_THRESHOLD,
    "reliable_minimum_completeness_percent":
        RELIABLE_MIN_COMPLETENESS,
    "normalized_duplicate_aggregation":
        NORMALIZED_DUPLICATE_AGGREGATION,
    "matched_common_nutrients": int(
        len(nutrient_comparison_df)
    ),
    "reliable_common_nutrients": int(
        len(reliable_common_nutrients_df)
    ),
    "normalized_reference_rows": int(
        len(usda_reference_df)
    ),
    "quality_reference_rows": int(
        len(usda_reference_quality_df)
    ),
    "validation_passed": bool(
        validation_checks_df["passed"].all()
    ),
    "generated_files": sorted(
        set(generated_files + [manifest_filename])
    ),
}

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
    )

print("Saved outputs:", len(manifest["generated_files"]))
print("Manifest:", manifest_path)
display(export_summary_df)

Saved outputs: 36
Manifest: C:\Github\Ceylon-Savour\data\processed\international\usda\usda_analysis_manifest.json


,filename,rows,status,path
0,usda_release_status.csv,2,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_release_status.csv
1,usda_file_inventory.csv,48,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_file_inventory.csv
2,usda_inventory_summary.csv,14,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_inventory_summary.csv
3,usda_csv_schema_summary.csv,36,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_csv_schema_summary.csv
4,usda_csv_schema_long.csv,182,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_csv_schema_long.csv
5,usda_table_role_mapping.csv,16,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_table_role_mapping.csv
6,usda_relationship_schema_validation.csv,11,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_relationship_schema_validation.csv
7,usda_relationship_validation.csv,11,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_relationship_validation.csv
8,usda_relationship_coverage.csv,11,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_relationship_coverage.csv
9,usda_foundation_record_type_summary.csv,5,saved,C:\Github\Ceylon-Savour\data\processed\international\usda\usda_foundation_record_type_summary.csv


## 19. USDA conclusion

The USDA analysis is complete when every row in `usda_validation_checks.csv`
passes.

The corrected workflow establishes that:

- Foundation results use only actual Foundation Food records rather than every
  row in the generic release `food.csv`.
- FNDDS nutrient identifiers are matched through `nutrient.nutrient_nbr`,
  while the nutrient master IDs are preserved separately.
- Relationship reports measure actual value coverage.
- Nutrient-master size and observed nutrient counts are reported separately.
- FNDDS `input_food.csv` is used as a partial international ingredient reference.
- Duplicate Foundation nutrient combinations are audited rather than removed.
- All generated files are recreated in one run and declared exactly in the manifest.

The next phase can begin with FlavorDB:
ingredient/entity → flavor molecule → taste and odour descriptor.

# Phase 3 — FlavorDB2 Analysis

## Phase 3A — Access, Licensing and Data Structure

This section examines how FlavorDB2 represents ingredients, flavor molecules,
taste descriptors, odour descriptors and ingredient–molecule relationships.

FlavorDB2 is used as a non-commercial academic reference for constructing
CeylonSavour ingredient and dish taste profiles.

#### Create FlavorDB folders

In [30]:
FLAVORDB_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "international"
    / "flavordb2"
)

FLAVORDB_EXPORT_DIR = FLAVORDB_ROOT / "manual_exports"

PROCESSED_FLAVORDB_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "international"
    / "flavordb2"
)

FLAVORDB_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_FLAVORDB_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Raw FlavorDB directory:")
print(FLAVORDB_EXPORT_DIR)

print("\nProcessed FlavorDB directory:")
print(PROCESSED_FLAVORDB_ROOT)

Raw FlavorDB directory:
C:\Github\Ceylon-Savour\data\raw\international\flavordb2\manual_exports

Processed FlavorDB directory:
C:\Github\Ceylon-Savour\data\processed\international\flavordb2


Record source and licensing metadata


In [31]:
flavordb_metadata = {
    "database_name": "FlavorDB2",
    "provider": "CoSyLab, IIIT-Delhi",
    "website": "https://cosylab.iiitd.edu.in/flavordb2/",
    "academic_use": True,
    "commercial_use": False,
    "license": (
        "Creative Commons Attribution-NonCommercial-"
        "ShareAlike 3.0 Unported"
    ),
    "attribution_required": True,
    "main_entities": [
        "ingredient",
        "natural_source",
        "flavor_molecule",
        "taste_descriptor",
        "odor_descriptor",
        "functional_group",
    ],
}

metadata_path = (
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_source_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        flavordb_metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

display(
    pd.DataFrame(
        flavordb_metadata.items(),
        columns=["field", "value"],
    )
)

,field,value
0,database_name,FlavorDB2
1,provider,"CoSyLab, IIIT-Delhi"
2,website,https://cosylab.iiitd.edu.in/flavordb2/
3,academic_use,True
4,commercial_use,False
5,license,Creative Commons Attribution-NonCommercial-ShareAlike 3.0 Unported
6,attribution_required,True
7,main_entities,"[ingredient, natural_source, flavor_molecule, taste_descriptor, odor_descriptor, functional_group]"


Select the first representative ingredients

In [32]:
FLAVORDB_SAMPLE_INGREDIENTS = [
    "coconut",
    "cinnamon",
    "black pepper",
    "cardamom",
    "clove",
    "ginger",
    "garlic",
    "onion",
    "chili pepper",
    "tamarind",
]

In [33]:
flavordb_download_log_df = pd.DataFrame({
    "ingredient_query": FLAVORDB_SAMPLE_INGREDIENTS,
    "flavordb_entity_id": pd.NA,
    "matched_entity_name": pd.NA,
    "category": pd.NA,
    "downloaded": False,
    "local_filename": pd.NA,
    "notes": pd.NA,
})

display(flavordb_download_log_df)

,ingredient_query,flavordb_entity_id,matched_entity_name,category,downloaded,local_filename,notes
0,coconut,<NA>,<NA>,<NA>,False,<NA>,<NA>
1,cinnamon,<NA>,<NA>,<NA>,False,<NA>,<NA>
2,black pepper,<NA>,<NA>,<NA>,False,<NA>,<NA>
3,cardamom,<NA>,<NA>,<NA>,False,<NA>,<NA>
4,clove,<NA>,<NA>,<NA>,False,<NA>,<NA>
5,ginger,<NA>,<NA>,<NA>,False,<NA>,<NA>
6,garlic,<NA>,<NA>,<NA>,False,<NA>,<NA>
7,onion,<NA>,<NA>,<NA>,False,<NA>,<NA>
8,chili pepper,<NA>,<NA>,<NA>,False,<NA>,<NA>
9,tamarind,<NA>,<NA>,<NA>,False,<NA>,<NA>


In [34]:
flavordb_download_log_path = (
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_entity_download_log.csv"
)

flavordb_download_log_df.to_csv(
    flavordb_download_log_path,
    index=False,
    encoding="utf-8-sig",
)

Inventory the downloaded JSON files


In [35]:
flavordb_json_files = sorted(
    FLAVORDB_EXPORT_DIR.glob("*.json")
)

flavordb_file_inventory_df = pd.DataFrame([
    {
        "filename": file.name,
        "size_kb": round(
            file.stat().st_size / 1024,
            2,
        ),
        "path": str(
            file.relative_to(PROJECT_ROOT)
        ),
    }
    for file in flavordb_json_files
])

if flavordb_file_inventory_df.empty:
    print(
        "No FlavorDB JSON files found yet. "
        "Download the first sample files and rerun this cell."
    )
else:
    display(flavordb_file_inventory_df)

,filename,size_kb,path
0,cardamom.json,225.25,data\raw\international\flavordb2\manual_exports\cardamom.json
1,cinnamon.json,239.02,data\raw\international\flavordb2\manual_exports\cinnamon.json
2,clove.json,229.78,data\raw\international\flavordb2\manual_exports\clove.json
3,coconut.json,207.76,data\raw\international\flavordb2\manual_exports\coconut.json
4,ginger.json,324.11,data\raw\international\flavordb2\manual_exports\ginger.json


Inspect the JSON structure safely

In [36]:
def describe_json_structure(value, prefix="root"):
    rows = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{prefix}.{key}"

            rows.append({
                "path": child_path,
                "python_type": type(child).__name__,
                "item_count": (
                    len(child)
                    if isinstance(
                        child,
                        (dict, list)
                    )
                    else None
                ),
            })

            if isinstance(child, dict):
                rows.extend(
                    describe_json_structure(
                        child,
                        child_path,
                    )
                )

            elif isinstance(child, list) and child:
                rows.extend(
                    describe_json_structure(
                        child[0],
                        f"{child_path}[0]",
                    )
                )

    return rows

In [37]:
if flavordb_json_files:
    first_json_path = flavordb_json_files[0]

    with open(
        first_json_path,
        "r",
        encoding="utf-8",
    ) as file:
        first_flavordb_json = json.load(file)

    print("Inspected file:", first_json_path.name)

    flavordb_structure_df = pd.DataFrame(
        describe_json_structure(
            first_flavordb_json
        )
    )

    display(flavordb_structure_df)

Inspected file: cardamom.json


,path,python_type,item_count
0,root.entity_id,int,NaN
1,root.category,str,NaN
2,root.category_readable,str,NaN
3,root.entity_alias,str,NaN
4,root.entity_alias_basket,str,NaN
5,root.entity_alias_readable,str,NaN
6,root.entity_alias_synonyms,str,NaN
7,root.entity_alias_url,str,NaN
8,root.natural_source_name,str,NaN
9,root.natural_source_url,str,NaN


In [38]:
if flavordb_json_files:
    if isinstance(first_flavordb_json, dict):
        print(
            "Top-level keys:",
            list(first_flavordb_json.keys()),
        )
    else:
        print(
            "Top-level JSON type:",
            type(first_flavordb_json).__name__,
        )

Top-level keys: ['entity_id', 'category', 'category_readable', 'entity_alias', 'entity_alias_basket', 'entity_alias_readable', 'entity_alias_synonyms', 'entity_alias_url', 'natural_source_name', 'natural_source_url', 'molecules']


In [39]:
display(flavordb_file_inventory_df)

,filename,size_kb,path
0,cardamom.json,225.25,data\raw\international\flavordb2\manual_exports\cardamom.json
1,cinnamon.json,239.02,data\raw\international\flavordb2\manual_exports\cinnamon.json
2,clove.json,229.78,data\raw\international\flavordb2\manual_exports\clove.json
3,coconut.json,207.76,data\raw\international\flavordb2\manual_exports\coconut.json
4,ginger.json,324.11,data\raw\international\flavordb2\manual_exports\ginger.json


### Inspect FlavorDB JSON Structure

Confirm downloaded files

In [40]:
flavordb_json_files = sorted(
    FLAVORDB_EXPORT_DIR.glob("*.json")
)

print("JSON files found:", len(flavordb_json_files))

for file in flavordb_json_files:
    print("-", file.name)

JSON files found: 5
- cardamom.json
- cinnamon.json
- clove.json
- coconut.json
- ginger.json


In [41]:
def load_json_file(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


if not flavordb_json_files:
    raise FileNotFoundError(
        f"No JSON files found in: {FLAVORDB_EXPORT_DIR}"
    )

sample_json_path = flavordb_json_files[0]
sample_json = load_json_file(sample_json_path)

print("Loaded:", sample_json_path.name)
print("Root type:", type(sample_json).__name__)

Loaded: cardamom.json
Root type: dict


In [42]:
if isinstance(sample_json, dict):
    print("Top-level keys:")

    for key, value in sample_json.items():
        print(
            f"- {key}: "
            f"type={type(value).__name__}, "
            f"size={len(value) if isinstance(value, (list, dict)) else 'N/A'}"
        )

elif isinstance(sample_json, list):
    print("Root list length:", len(sample_json))

    if sample_json:
        print("First item type:", type(sample_json[0]).__name__)

        if isinstance(sample_json[0], dict):
            print("First item keys:")
            print(list(sample_json[0].keys()))

Top-level keys:
- entity_id: type=int, size=N/A
- category: type=str, size=N/A
- category_readable: type=str, size=N/A
- entity_alias: type=str, size=N/A
- entity_alias_basket: type=str, size=N/A
- entity_alias_readable: type=str, size=N/A
- entity_alias_synonyms: type=str, size=N/A
- entity_alias_url: type=str, size=N/A
- natural_source_name: type=str, size=N/A
- natural_source_url: type=str, size=N/A
- molecules: type=list, size=173


Create a compact structure report

In [43]:
def inspect_json(value, path="root", depth=0, max_depth=4):
    rows = []

    rows.append({
        "path": path,
        "type": type(value).__name__,
        "size": (
            len(value)
            if isinstance(value, (list, dict))
            else None
        ),
    })

    if depth >= max_depth:
        return rows

    if isinstance(value, dict):
        for key, child in value.items():
            rows.extend(
                inspect_json(
                    child,
                    path=f"{path}.{key}",
                    depth=depth + 1,
                    max_depth=max_depth,
                )
            )

    elif isinstance(value, list) and value:
        rows.extend(
            inspect_json(
                value[0],
                path=f"{path}[0]",
                depth=depth + 1,
                max_depth=max_depth,
            )
        )

    return rows


flavordb_structure_df = pd.DataFrame(
    inspect_json(sample_json)
)

display(flavordb_structure_df)

,path,type,size
0,root,dict,11.0
1,root.entity_id,int,NaN
2,root.category,str,NaN
3,root.category_readable,str,NaN
4,root.entity_alias,str,NaN
5,root.entity_alias_basket,str,NaN
6,root.entity_alias_readable,str,NaN
7,root.entity_alias_synonyms,str,NaN
8,root.entity_alias_url,str,NaN
9,root.natural_source_name,str,NaN


Save the structure report

In [44]:
structure_output_path = (
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_json_structure.csv"
)

flavordb_structure_df.to_csv(
    structure_output_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", structure_output_path)

Saved: C:\Github\Ceylon-Savour\data\processed\international\flavordb2\flavordb_json_structure.csv


### Inspect Molecule Fields

Inspect one molecule record

In [46]:
first_molecule = sample_json["molecules"][0]

print("Molecule field count:", len(first_molecule))
print("Molecule fields:")

for key in first_molecule.keys():
    print("-", key)

Molecule field count: 43
Molecule fields:
- pubchem_id
- iupac_name
- common_name
- smile
- molecular_weight
- hbd_count
- hba_count
- num_rotatablebonds
- complexity
- topological_polor_surfacearea
- monoisotopic_mass
- exact_mass
- xlogp
- charge
- heavy_atom_count
- atom_stereo_count
- defined_atom_stereocenter_count
- undefined_atom_stereocenter_count
- bond_stereo_count
- defined_bond_stereocenter_count
- undefined_bond_stereocenter_count
- isotope_atom_count
- covalently_bonded_unit_count
- cas_id
- fema_number
- fema_flavor_profile
- odor
- taste
- functional_groups
- inchi
- volume3d
- fooddb_flavor_profile
- super_sweet
- bitter
- supersweetdb_id
- bitterdb_id
- fooddb_id
- flavornet_id
- fenoroli_and_os
- natural
- unknown_natural
- synthetic
- flavor_profile


Create a molecule field summary

In [47]:
def preview_value(value, max_length=120):
    if isinstance(value, (dict, list)):
        value = json.dumps(
            value,
            ensure_ascii=False,
        )

    value = str(value)

    return (
        value[:max_length] + "..."
        if len(value) > max_length
        else value
    )


field_summary_rows = []

for field, value in first_molecule.items():
    field_summary_rows.append({
        "field": field,
        "type": type(value).__name__,
        "sample_value": preview_value(value),
    })

molecule_field_summary_df = pd.DataFrame(
    field_summary_rows
)

display(molecule_field_summary_df)

,field,type,sample_value
0,pubchem_id,int,323
1,iupac_name,str,chromen-2-one
2,common_name,str,coumarin
3,smile,str,C1=CC=C2C(=C1)C=CC(=O)O2
4,molecular_weight,float,146.145
5,hbd_count,int,0
6,hba_count,int,2
7,num_rotatablebonds,int,0
8,complexity,float,196.0
9,topological_polor_surfacearea,float,26.3


Show only useful taste and aroma fields

In [48]:
descriptor_keywords = [
    "name",
    "taste",
    "odor",
    "odour",
    "flavor",
    "flavour",
    "aroma",
    "functional",
    "pubchem",
    "smile",
    "cas",
]

important_molecule_fields_df = (
    molecule_field_summary_df[
        molecule_field_summary_df["field"]
        .str.lower()
        .apply(
            lambda field: any(
                keyword in field
                for keyword in descriptor_keywords
            )
        )
    ]
    .reset_index(drop=True)
)

display(important_molecule_fields_df)

,field,type,sample_value
0,pubchem_id,int,323
1,iupac_name,str,chromen-2-one
2,common_name,str,coumarin
3,smile,str,C1=CC=C2C(=C1)C=CC(=O)O2
4,cas_id,str,91-64-5
5,fema_flavor_profile,NoneType,None
6,odor,str,"pleasant, fragrant odor resembling that of vanilla beans.@hay-like bittersweet odor"
7,taste,str,"bitter undertone@nut-like flavor on dilution@bitter, aromatic@burning taste"
8,functional_groups,str,oxo(het)arene@aromatic compound@heterocyclic compound
9,fooddb_flavor_profile,str,new mown hay@bitter@green@sweet@tonka


Inspect all downloaded ingredients

In [49]:
ingredient_summary_rows = []

for json_path in flavordb_json_files:
    data = load_json_file(json_path)

    ingredient_summary_rows.append({
        "filename": json_path.name,
        "entity_id": data.get("entity_id"),
        "ingredient_name": (
            data.get("natural_source_name")
            or data.get("entity_alias_readable")
            or data.get("entity_alias")
        ),
        "category": (
            data.get("category_readable")
            or data.get("category")
        ),
        "molecule_count": len(
            data.get("molecules", [])
        ),
    })

flavordb_ingredient_summary_df = pd.DataFrame(
    ingredient_summary_rows
)

display(flavordb_ingredient_summary_df)

,filename,entity_id,ingredient_name,category,molecule_count
0,cardamom.json,327,Elattaria,Spice,173
1,cinnamon.json,330,Cinnamomum,Spice,183
2,clove.json,331,Syzygium,Spice,177
3,coconut.json,172,Cocos Nucifera,Fruit,161
4,ginger.json,333,Zingiber,Spice,251


In [50]:
molecule_field_summary_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_molecule_field_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

flavordb_ingredient_summary_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_ingredient_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

## Normalize FlavorDB Ingredient–Molecule Data

In [51]:
def first_available(record, candidate_fields, default=None):
    for field in candidate_fields:
        if field in record:
            value = record.get(field)

            if value not in [None, "", [], {}]:
                return value

    return default


def normalize_descriptor_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    if isinstance(value, dict):
        values = []

        for item in value.values():
            if isinstance(item, list):
                values.extend(item)
            else:
                values.append(item)

        return [
            str(item).strip()
            for item in values
            if str(item).strip()
        ]

    text = str(value).strip()

    if not text:
        return []

    for separator in [";", "|", ","]:
        if separator in text:
            return [
                item.strip()
                for item in text.split(separator)
                if item.strip()
            ]

    return [text]

Parse all ingredient JSON files

In [52]:
flavordb_molecule_rows = []

for json_path in flavordb_json_files:
    data = load_json_file(json_path)

    entity_id = data.get("entity_id")

    ingredient_name = (
        data.get("natural_source_name")
        or data.get("entity_alias_readable")
        or data.get("entity_alias")
        or json_path.stem
    )

    category = (
        data.get("category_readable")
        or data.get("category")
    )

    molecules = data.get("molecules", [])

    for molecule in molecules:
        taste_value = first_available(
            molecule,
            [
                "taste",
                "taste_profile",
                "taste_descriptors",
            ],
        )

        odor_value = first_available(
            molecule,
            [
                "odor",
                "odour",
                "odor_profile",
                "odour_profile",
                "odor_descriptors",
                "aroma",
            ],
        )

        flavor_value = first_available(
            molecule,
            [
                "flavor",
                "flavour",
                "flavor_profile",
                "flavour_profile",
            ],
        )

        functional_group_value = first_available(
            molecule,
            [
                "functional_group",
                "functional_groups",
            ],
        )

        flavordb_molecule_rows.append({
            "entity_id": entity_id,
            "ingredient_name": ingredient_name,
            "category": category,

            "molecule_pubchem_id": first_available(
                molecule,
                [
                    "pubchem_id",
                    "pubchem_cid",
                    "pubchem",
                ],
            ),

            "molecule_name": first_available(
                molecule,
                [
                    "common_name",
                    "molecule_name",
                    "iupac_name",
                    "name",
                ],
            ),

            "iupac_name": first_available(
                molecule,
                ["iupac_name"],
            ),

            "cas_number": first_available(
                molecule,
                [
                    "cas_number",
                    "cas_id",
                    "cas",
                ],
            ),

            "smiles": first_available(
                molecule,
                [
                    "canonical_smiles",
                    "smiles",
                    "isomeric_smiles",
                ],
            ),

            "taste_descriptors": normalize_descriptor_list(
                taste_value
            ),

            "odor_descriptors": normalize_descriptor_list(
                odor_value
            ),

            "flavor_descriptors": normalize_descriptor_list(
                flavor_value
            ),

            "functional_groups": normalize_descriptor_list(
                functional_group_value
            ),

            "source_file": json_path.name,
        })

Create the molecule table

In [53]:
flavordb_molecule_df = pd.DataFrame(
    flavordb_molecule_rows
)

print("Total molecule rows:", len(flavordb_molecule_df))
print(
    "Unique ingredients:",
    flavordb_molecule_df["entity_id"].nunique(),
)

display(flavordb_molecule_df.head())

Total molecule rows: 945
Unique ingredients: 5


,entity_id,ingredient_name,category,molecule_pubchem_id,molecule_name,iupac_name,cas_number,smiles,taste_descriptors,odor_descriptors,flavor_descriptors,functional_groups,source_file
0,327,Elattaria,Spice,323,coumarin,chromen-2-one,91-64-5,None,"[bitter undertone@nut-like flavor on dilution@bitter, aromatic@burning taste]","[pleasant, fragrant odor resembling that of vanilla beans.@hay-like bittersweet odor]",[sweet@new mown hay@green@tonka@bitter],[oxo(het)arene@aromatic compound@heterocyclic compound],cardamom.json
1,327,Elattaria,Spice,6989,Thymol,5-methyl-2-propan-2-ylphenol,89-83-8,None,"[pungent, caustic@sweet, medicinal, spicy@aromatic taste]","[odor of thyme@a spicy-herbal, slightly medicinal odor reminiscent of thyme@aromatic odor]",[herbal@thyme@camphor@medicinal@phenolic],[hydroxy compound@phenol or hydroxyhetarene@aromatic compound],cardamom.json
2,327,Elattaria,Spice,7150,Methyl Benzoate,methyl benzoate,93-58-3,None,"[fruity, nutty, ... with cherry taste]","[fragrant odor@fruity odor, also similar to cananga@medicinal odor]",[prune@cananga@floral@wintergreen@lettuce@sweet@almond@herb@phenolic],[carboxylic acid derivative@carboxylic acid ester@aromatic compound],cardamom.json
3,327,Elattaria,Spice,8468,Vanillic acid,4-hydroxy-3-methoxybenzoic acid,121-34-6,None,[],[],[powdery@vanilla@bean@milky@sweet@creamy@dairy],[hydroxy compound@phenol or hydroxyhetarene@ether@alkyl aryl ether@carboxylic acid derivative@carboxylic acid@aromatic compound],cardamom.json
4,327,Elattaria,Spice,442355,AC1L9CNW,NaN,3856-25-5,None,[],[],[wood@woody@spice],[alkene],cardamom.json


In [54]:
descriptor_summary_df = pd.DataFrame([
    {
        "field": "taste_descriptors",
        "rows_with_values": (
            flavordb_molecule_df[
                "taste_descriptors"
            ].map(len) > 0
        ).sum(),
    },
    {
        "field": "odor_descriptors",
        "rows_with_values": (
            flavordb_molecule_df[
                "odor_descriptors"
            ].map(len) > 0
        ).sum(),
    },
    {
        "field": "flavor_descriptors",
        "rows_with_values": (
            flavordb_molecule_df[
                "flavor_descriptors"
            ].map(len) > 0
        ).sum(),
    },
    {
        "field": "functional_groups",
        "rows_with_values": (
            flavordb_molecule_df[
                "functional_groups"
            ].map(len) > 0
        ).sum(),
    },
])

descriptor_summary_df["coverage_percent"] = (
    descriptor_summary_df["rows_with_values"]
    / len(flavordb_molecule_df)
    * 100
).round(2)

display(descriptor_summary_df)

,field,rows_with_values,coverage_percent
0,taste_descriptors,333,35.24
1,odor_descriptors,441,46.67
2,flavor_descriptors,930,98.41
3,functional_groups,931,98.52


Taste descriptors

In [55]:
flavordb_taste_long_df = (
    flavordb_molecule_df[
        [
            "entity_id",
            "ingredient_name",
            "molecule_pubchem_id",
            "molecule_name",
            "taste_descriptors",
        ]
    ]
    .explode("taste_descriptors")
    .rename(columns={
        "taste_descriptors": "taste_descriptor"
    })
)

flavordb_taste_long_df = flavordb_taste_long_df[
    flavordb_taste_long_df[
        "taste_descriptor"
    ].notna()
].reset_index(drop=True)

display(flavordb_taste_long_df.head())

,entity_id,ingredient_name,molecule_pubchem_id,molecule_name,taste_descriptor
0,327,Elattaria,323,coumarin,bitter undertone@nut-like flavor on dilution@bitter
1,327,Elattaria,323,coumarin,aromatic@burning taste
2,327,Elattaria,6989,Thymol,pungent
3,327,Elattaria,6989,Thymol,caustic@sweet
4,327,Elattaria,6989,Thymol,medicinal


Odour descriptors

In [56]:
flavordb_odor_long_df = (
    flavordb_molecule_df[
        [
            "entity_id",
            "ingredient_name",
            "molecule_pubchem_id",
            "molecule_name",
            "odor_descriptors",
        ]
    ]
    .explode("odor_descriptors")
    .rename(columns={
        "odor_descriptors": "odor_descriptor"
    })
)

flavordb_odor_long_df = flavordb_odor_long_df[
    flavordb_odor_long_df[
        "odor_descriptor"
    ].notna()
].reset_index(drop=True)

display(flavordb_odor_long_df.head())

,entity_id,ingredient_name,molecule_pubchem_id,molecule_name,odor_descriptor
0,327,Elattaria,323,coumarin,pleasant
1,327,Elattaria,323,coumarin,fragrant odor resembling that of vanilla beans.@hay-like bittersweet odor
2,327,Elattaria,6989,Thymol,odor of thyme@a spicy-herbal
3,327,Elattaria,6989,Thymol,slightly medicinal odor reminiscent of thyme@aromatic odor
4,327,Elattaria,7150,Methyl Benzoate,fragrant odor@fruity odor


In [57]:
flavordb_molecule_df.to_json(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_ingredient_molecule_normalized.json",
    orient="records",
    indent=2,
    force_ascii=False,
)

flavordb_taste_long_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_taste_descriptors.csv",
    index=False,
    encoding="utf-8-sig",
)

flavordb_odor_long_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_odor_descriptors.csv",
    index=False,
    encoding="utf-8-sig",
)

descriptor_summary_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_descriptor_coverage.csv",
    index=False,
    encoding="utf-8-sig",
)

print("FlavorDB normalized outputs saved.")

FlavorDB normalized outputs saved.


## Ingredient-Level Taste and Aroma Profiles

In [58]:
def clean_descriptor(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )


if not flavordb_taste_long_df.empty:
    flavordb_taste_long_df["taste_descriptor"] = clean_descriptor(
        flavordb_taste_long_df["taste_descriptor"]
    )

if not flavordb_odor_long_df.empty:
    flavordb_odor_long_df["odor_descriptor"] = clean_descriptor(
        flavordb_odor_long_df["odor_descriptor"]
    )

Count descriptors per ingredient

In [59]:
taste_frequency_df = (
    flavordb_taste_long_df
    .dropna(subset=["taste_descriptor"])
    .groupby(
        [
            "entity_id",
            "ingredient_name",
            "taste_descriptor",
        ]
    )
    .size()
    .reset_index(name="molecule_count")
    .sort_values(
        [
            "ingredient_name",
            "molecule_count",
        ],
        ascending=[True, False],
    )
)

odor_frequency_df = (
    flavordb_odor_long_df
    .dropna(subset=["odor_descriptor"])
    .groupby(
        [
            "entity_id",
            "ingredient_name",
            "odor_descriptor",
        ]
    )
    .size()
    .reset_index(name="molecule_count")
    .sort_values(
        [
            "ingredient_name",
            "molecule_count",
        ],
        ascending=[True, False],
    )
)

display(taste_frequency_df.head(20))
display(odor_frequency_df.head(20))

,entity_id,ingredient_name,taste_descriptor,molecule_count
227,330,Cinnamomum,bitter,7
229,330,Cinnamomum,bitter taste,3
234,330,Cinnamomum,burning taste,3
263,330,Cinnamomum,fruity,3
289,330,Cinnamomum,pungent,3
302,330,Cinnamomum,sweet,3
324,330,Cinnamomum,woody,3
214,330,Cinnamomum,30 to 460 ppb.@characteristic bittersweet,2
215,330,Cinnamomum,32 to 460 ppb@beta-citral,2
246,330,Cinnamomum,citrus,2


,entity_id,ingredient_name,odor_descriptor,molecule_count
314,330,Cinnamomum,floral odor,3
353,330,Cinnamomum,odorless,3
373,330,Cinnamomum,pungent,3
284,330,Cinnamomum,camphor-like odor,2
288,330,Cinnamomum,characteristic,2
311,330,Cinnamomum,fatty,2
313,330,Cinnamomum,floral,2
332,330,Cinnamomum,green,2
346,330,Cinnamomum,odor of hyacinth@pleasant,2
367,330,Cinnamomum,piney,2


Define CeylonSavour taste mappings

In [60]:
TASTE_KEYWORDS = {
    "sweetness": {
        "sweet",
        "sugary",
        "honey",
        "caramel",
        "vanilla",
    },
    "sourness": {
        "sour",
        "acidic",
        "tart",
        "vinegar",
    },
    "bitterness": {
        "bitter",
    },
    "umami": {
        "umami",
        "savory",
        "savoury",
        "meaty",
        "brothy",
    },
    "spice_base": {
        "pungent",
        "spicy",
        "peppery",
        "hot",
        "warming",
    },
    "saltiness": {
        "salty",
        "salt",
    },
}

Score each ingredient

In [61]:
def descriptor_matches(descriptor, keywords):
    descriptor = str(descriptor).lower()

    return any(
        keyword in descriptor
        for keyword in keywords
    )


ingredient_taste_rows = []

for (
    entity_id,
    ingredient_name,
), group in taste_frequency_df.groupby(
    ["entity_id", "ingredient_name"]
):
    total_count = group["molecule_count"].sum()

    row = {
        "entity_id": entity_id,
        "ingredient_name": ingredient_name,
        "total_taste_links": total_count,
    }

    for taste_field, keywords in TASTE_KEYWORDS.items():
        matched_count = group.loc[
            group["taste_descriptor"].apply(
                lambda value: descriptor_matches(
                    value,
                    keywords,
                )
            ),
            "molecule_count",
        ].sum()

        row[taste_field] = round(
            matched_count / total_count,
            4,
        ) if total_count else 0.0

    ingredient_taste_rows.append(row)


ingredient_taste_profile_df = pd.DataFrame(
    ingredient_taste_rows
)

display(ingredient_taste_profile_df)

,entity_id,ingredient_name,total_taste_links,sweetness,sourness,bitterness,umami,spice_base,saltiness
0,172,Cocos Nucifera,110,0.1545,0.0273,0.1455,0.0,0.0273,0.0
1,327,Elattaria,147,0.1905,0.0204,0.1224,0.0,0.0952,0.0
2,330,Cinnamomum,140,0.2000,0.0071,0.1500,0.0,0.1143,0.0
3,331,Syzygium,123,0.1951,0.0081,0.1789,0.0,0.0813,0.0
4,333,Zingiber,158,0.2025,0.0127,0.1203,0.0,0.0886,0.0


Extract top aroma tags

In [62]:
def top_descriptors(group, descriptor_column, top_n=3):
    return (
        group
        .sort_values(
            "molecule_count",
            ascending=False,
        )
        [descriptor_column]
        .dropna()
        .drop_duplicates()
        .head(top_n)
        .tolist()
    )


ingredient_aroma_rows = []

for (
    entity_id,
    ingredient_name,
), group in odor_frequency_df.groupby(
    ["entity_id", "ingredient_name"]
):
    ingredient_aroma_rows.append({
        "entity_id": entity_id,
        "ingredient_name": ingredient_name,
        "top_aroma_tags": top_descriptors(
            group,
            "odor_descriptor",
            top_n=3,
        ),
        "unique_odor_descriptors":
            group["odor_descriptor"].nunique(),
    })


ingredient_aroma_profile_df = pd.DataFrame(
    ingredient_aroma_rows
)

display(ingredient_aroma_profile_df)

,entity_id,ingredient_name,top_aroma_tags,unique_odor_descriptors
0,172,Cocos Nucifera,"[odorless, characteristic, powerful]",125
1,327,Elattaria,"[odorless, characteristic, fatty]",143
2,330,Cinnamomum,"[floral odor, odorless, pungent]",147
3,331,Syzygium,"[fatty, floral odor, odorless]",130
4,333,Zingiber,"[characteristic, sweet, camphor-like odor]",164


Combine taste and aroma profiles

In [63]:
flavordb_ingredient_profile_df = (
    flavordb_ingredient_summary_df
    .merge(
        ingredient_taste_profile_df,
        on=[
            "entity_id",
            "ingredient_name",
        ],
        how="left",
    )
    .merge(
        ingredient_aroma_profile_df,
        on=[
            "entity_id",
            "ingredient_name",
        ],
        how="left",
    )
)

score_columns = list(TASTE_KEYWORDS.keys())

flavordb_ingredient_profile_df[
    score_columns
] = flavordb_ingredient_profile_df[
    score_columns
].fillna(0.0)

flavordb_ingredient_profile_df[
    "top_aroma_tags"
] = flavordb_ingredient_profile_df[
    "top_aroma_tags"
].apply(
    lambda value: value
    if isinstance(value, list)
    else []
)

display(flavordb_ingredient_profile_df)

,filename,entity_id,ingredient_name,category,molecule_count,total_taste_links,sweetness,sourness,bitterness,umami,spice_base,saltiness,top_aroma_tags,unique_odor_descriptors
0,cardamom.json,327,Elattaria,Spice,173,147,0.1905,0.0204,0.1224,0.0,0.0952,0.0,"[odorless, characteristic, fatty]",143
1,cinnamon.json,330,Cinnamomum,Spice,183,140,0.2000,0.0071,0.1500,0.0,0.1143,0.0,"[floral odor, odorless, pungent]",147
2,clove.json,331,Syzygium,Spice,177,123,0.1951,0.0081,0.1789,0.0,0.0813,0.0,"[fatty, floral odor, odorless]",130
3,coconut.json,172,Cocos Nucifera,Fruit,161,110,0.1545,0.0273,0.1455,0.0,0.0273,0.0,"[odorless, characteristic, powerful]",125
4,ginger.json,333,Zingiber,Spice,251,158,0.2025,0.0127,0.1203,0.0,0.0886,0.0,"[characteristic, sweet, camphor-like odor]",164


In [64]:
taste_frequency_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_taste_frequency.csv",
    index=False,
    encoding="utf-8-sig",
)

odor_frequency_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_odor_frequency.csv",
    index=False,
    encoding="utf-8-sig",
)

flavordb_ingredient_profile_df.to_json(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_ingredient_profiles.json",
    orient="records",
    indent=2,
    force_ascii=False,
)

print("Ingredient taste and aroma profiles saved.")

Ingredient taste and aroma profiles saved.


## Data Quality and CeylonSavour Field Mapping

Prepare ingredient profiles

In [65]:
taste_score_columns = [
    "sweetness",
    "sourness",
    "bitterness",
    "umami",
    "spice_base",
    "saltiness",
]

flavordb_quality_df = (
    flavordb_ingredient_profile_df.copy()
)

for column in taste_score_columns:
    if column not in flavordb_quality_df.columns:
        flavordb_quality_df[column] = 0.0

flavordb_quality_df[taste_score_columns] = (
    flavordb_quality_df[taste_score_columns]
    .fillna(0.0)
)

flavordb_quality_df["total_taste_links"] = (
    flavordb_quality_df
    .get("total_taste_links", 0)
    .fillna(0)
)

flavordb_quality_df["unique_odor_descriptors"] = (
    flavordb_quality_df
    .get("unique_odor_descriptors", 0)
    .fillna(0)
)

Calculate data-quality score

In [66]:
flavordb_quality_df["has_entity_id"] = (
    flavordb_quality_df["entity_id"].notna()
)

flavordb_quality_df["has_ingredient_name"] = (
    flavordb_quality_df["ingredient_name"]
    .notna()
)

flavordb_quality_df["has_molecules"] = (
    flavordb_quality_df["molecule_count"]
    .fillna(0)
    .gt(0)
)

flavordb_quality_df["has_taste_data"] = (
    flavordb_quality_df["total_taste_links"]
    .gt(0)
)

flavordb_quality_df["has_odor_data"] = (
    flavordb_quality_df[
        "unique_odor_descriptors"
    ].gt(0)
)

quality_columns = [
    "has_entity_id",
    "has_ingredient_name",
    "has_molecules",
    "has_taste_data",
    "has_odor_data",
]

flavordb_quality_df["data_quality_score"] = (
    flavordb_quality_df[quality_columns]
    .mean(axis=1)
    .round(2)
)

flavordb_quality_df["quality_level"] = pd.cut(
    flavordb_quality_df["data_quality_score"],
    bins=[-0.01, 0.49, 0.79, 1.0],
    labels=["low", "medium", "high"],
)

display(
    flavordb_quality_df[
        [
            "entity_id",
            "ingredient_name",
            "molecule_count",
            "total_taste_links",
            "unique_odor_descriptors",
            "data_quality_score",
            "quality_level",
        ]
    ]
)

,entity_id,ingredient_name,molecule_count,total_taste_links,unique_odor_descriptors,data_quality_score,quality_level
0,327,Elattaria,173,147,143,1.0,high
1,330,Cinnamomum,183,140,147,1.0,high
2,331,Syzygium,177,123,130,1.0,high
3,172,Cocos Nucifera,161,110,125,1.0,high
4,333,Zingiber,251,158,164,1.0,high


Check duplicate and missing records

In [67]:
quality_checks_df = pd.DataFrame([
    {
        "check": "Duplicate entity IDs",
        "value": (
            flavordb_quality_df["entity_id"]
            .duplicated()
            .sum()
        ),
    },
    {
        "check": "Missing ingredient names",
        "value": (
            flavordb_quality_df[
                "ingredient_name"
            ].isna().sum()
        ),
    },
    {
        "check": "Ingredients without molecules",
        "value": (
            ~flavordb_quality_df[
                "has_molecules"
            ]
        ).sum(),
    },
    {
        "check": "Ingredients without taste data",
        "value": (
            ~flavordb_quality_df[
                "has_taste_data"
            ]
        ).sum(),
    },
    {
        "check": "Ingredients without odor data",
        "value": (
            ~flavordb_quality_df[
                "has_odor_data"
            ]
        ).sum(),
    },
])

display(quality_checks_df)

,check,value
0,Duplicate entity IDs,0
1,Missing ingredient names,0
2,Ingredients without molecules,0
3,Ingredients without taste data,0
4,Ingredients without odor data,0


Create manual-review list

In [68]:
flavordb_manual_review_df = (
    flavordb_quality_df[
        flavordb_quality_df[
            "data_quality_score"
        ] < 0.8
    ]
    [
        [
            "entity_id",
            "ingredient_name",
            "category",
            "molecule_count",
            "total_taste_links",
            "unique_odor_descriptors",
            "data_quality_score",
            "quality_level",
        ]
    ]
    .reset_index(drop=True)
)

display(flavordb_manual_review_df)

,entity_id,ingredient_name,category,molecule_count,total_taste_links,unique_odor_descriptors,data_quality_score,quality_level


In [70]:
print(
    flavordb_quality_df["data_quality_score"]
    .value_counts()
    .sort_index()
)

print(
    "Manual review rows:",
    len(flavordb_manual_review_df)
)

data_quality_score
1.0    5
Name: count, dtype: int64
Manual review rows: 0


Map to CeylonSavour fields

In [71]:
ceylonsavour_flavor_mapping_df = (
    flavordb_quality_df[
        [
            "entity_id",
            "ingredient_name",
            "category",
            "molecule_count",
            "total_taste_links",
            "unique_odor_descriptors",
            *taste_score_columns,
            "top_aroma_tags",
            "data_quality_score",
            "quality_level",
        ]
    ]
    .rename(columns={
        "entity_id": "source_entity_id",
        "ingredient_name": "canonical_ingredient_name",
        "category": "source_category",
        "molecule_count": "flavor_molecule_count",
        "total_taste_links": "taste_evidence_count",
        "unique_odor_descriptors":
            "odor_descriptor_count",
        "top_aroma_tags": "aroma_tags",
    })
)

ceylonsavour_flavor_mapping_df[
    "source_dataset"
] = "flavordb2"

ceylonsavour_flavor_mapping_df[
    "source_license"
] = "CC BY-NC-SA 3.0"

ceylonsavour_flavor_mapping_df[
    "profile_method"
] = "descriptor-frequency mapping"

display(ceylonsavour_flavor_mapping_df.head())

,source_entity_id,canonical_ingredient_name,source_category,flavor_molecule_count,taste_evidence_count,odor_descriptor_count,sweetness,sourness,bitterness,umami,spice_base,saltiness,aroma_tags,data_quality_score,quality_level,source_dataset,source_license,profile_method
0,327,Elattaria,Spice,173,147,143,0.1905,0.0204,0.1224,0.0,0.0952,0.0,"[odorless, characteristic, fatty]",1.0,high,flavordb2,CC BY-NC-SA 3.0,descriptor-frequency mapping
1,330,Cinnamomum,Spice,183,140,147,0.2000,0.0071,0.1500,0.0,0.1143,0.0,"[floral odor, odorless, pungent]",1.0,high,flavordb2,CC BY-NC-SA 3.0,descriptor-frequency mapping
2,331,Syzygium,Spice,177,123,130,0.1951,0.0081,0.1789,0.0,0.0813,0.0,"[fatty, floral odor, odorless]",1.0,high,flavordb2,CC BY-NC-SA 3.0,descriptor-frequency mapping
3,172,Cocos Nucifera,Fruit,161,110,125,0.1545,0.0273,0.1455,0.0,0.0273,0.0,"[odorless, characteristic, powerful]",1.0,high,flavordb2,CC BY-NC-SA 3.0,descriptor-frequency mapping
4,333,Zingiber,Spice,251,158,164,0.2025,0.0127,0.1203,0.0,0.0886,0.0,"[characteristic, sweet, camphor-like odor]",1.0,high,flavordb2,CC BY-NC-SA 3.0,descriptor-frequency mapping


In [72]:
flavordb_quality_export_df = (
    flavordb_quality_df.copy()
)

flavordb_quality_export_df[
    "top_aroma_tags"
] = flavordb_quality_export_df[
    "top_aroma_tags"
].apply(
    lambda value: json.dumps(
        value if isinstance(value, list) else [],
        ensure_ascii=False,
    )
)

ceylonsavour_export_df = (
    ceylonsavour_flavor_mapping_df.copy()
)

ceylonsavour_export_df[
    "aroma_tags"
] = ceylonsavour_export_df[
    "aroma_tags"
].apply(
    lambda value: json.dumps(
        value if isinstance(value, list) else [],
        ensure_ascii=False,
    )
)

flavordb_quality_export_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_ingredient_quality.csv",
    index=False,
    encoding="utf-8-sig",
)

quality_checks_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_quality_checks.csv",
    index=False,
    encoding="utf-8-sig",
)

flavordb_manual_review_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_manual_review.csv",
    index=False,
    encoding="utf-8-sig",
)

ceylonsavour_export_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_ceylonsavour_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

print("FlavorDB quality and mapping outputs saved.")

FlavorDB quality and mapping outputs saved.


## Final Findings and Limitations

Create summary metrics

In [73]:
flavordb_summary_df = pd.DataFrame([
    {
        "metric": "Downloaded ingredients",
        "value": flavordb_ingredient_summary_df[
            "entity_id"
        ].nunique(),
    },
    {
        "metric": "Total molecule records",
        "value": len(flavordb_molecule_df),
    },
    {
        "metric": "Taste descriptor records",
        "value": len(flavordb_taste_long_df),
    },
    {
        "metric": "Odor descriptor records",
        "value": len(flavordb_odor_long_df),
    },
    {
        "metric": "High-quality ingredient profiles",
        "value": (
            flavordb_quality_df["quality_level"]
            .astype(str)
            .eq("high")
            .sum()
        ),
    },
    {
        "metric": "Manual-review ingredients",
        "value": len(flavordb_manual_review_df),
    },
])

display(flavordb_summary_df)

,metric,value
0,Downloaded ingredients,5
1,Total molecule records,945
2,Taste descriptor records,678
3,Odor descriptor records,829
4,High-quality ingredient profiles,5
5,Manual-review ingredients,0



Create final findings

In [74]:
flavordb_final_findings_df = pd.DataFrame([
    {
        "area": "Ingredient representation",
        "finding": (
            "FlavorDB represents natural ingredients as entities "
            "linked to multiple flavor molecules."
        ),
    },
    {
        "area": "Taste information",
        "finding": (
            "Taste descriptors can support ingredient-level "
            "sweetness, sourness, bitterness, umami, spice, "
            "and saltiness evidence."
        ),
    },
    {
        "area": "Aroma information",
        "finding": (
            "Odor descriptors provide useful aroma tags for "
            "ingredient and dish profiles."
        ),
    },
    {
        "area": "CeylonSavour use",
        "finding": (
            "FlavorDB can enrich ingredient taste and aroma "
            "features before dish-level aggregation."
        ),
    },
    {
        "area": "Limitation",
        "finding": (
            "Descriptor frequency is an evidence score and must "
            "not be treated as an exact sensory intensity."
        ),
    },
    {
        "area": "Sri Lankan ingredients",
        "finding": (
            "Ingredients missing from FlavorDB require manual "
            "annotation and expert validation."
        ),
    },
])

display(flavordb_final_findings_df)

,area,finding
0,Ingredient representation,FlavorDB represents natural ingredients as entities linked to multiple flavor molecules.
1,Taste information,"Taste descriptors can support ingredient-level sweetness, sourness, bitterness, umami, spice, and saltiness evidence."
2,Aroma information,Odor descriptors provide useful aroma tags for ingredient and dish profiles.
3,CeylonSavour use,FlavorDB can enrich ingredient taste and aroma features before dish-level aggregation.
4,Limitation,Descriptor frequency is an evidence score and must not be treated as an exact sensory intensity.
5,Sri Lankan ingredients,Ingredients missing from FlavorDB require manual annotation and expert validation.


## FlavorDB2 Conclusion

The FlavorDB2 analysis confirmed that the database provides a useful
ingredient-centred structure:

**ingredient → flavor molecule → taste and odor descriptors**

For CeylonSavour, the most useful information is:

- ingredient identity and category;
- associated flavor molecules;
- taste descriptors;
- odor and aroma descriptors;
- functional-group information;
- source identifiers for provenance.

The extracted descriptors were normalized into ingredient-level taste
and aroma profiles. These profiles can later support Sri Lankan dish
vectors by combining the profiles of their ingredients.

However, the generated taste scores represent relative descriptor
evidence, not experimentally measured sensory intensity. They must be
combined with recipe quantities, ingredient importance, culinary
knowledge and expert validation.

This analysis currently uses a small representative sample of downloaded
ingredients. It demonstrates the structure and processing pipeline, but
does not represent complete FlavorDB coverage.

FlavorDB2 data must be attributed and used according to its
CC BY-NC-SA 3.0 non-commercial licence.

In [75]:
flavordb_summary_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_analysis_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

flavordb_final_findings_df.to_csv(
    PROCESSED_FLAVORDB_ROOT
    / "flavordb_final_findings.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Final FlavorDB findings saved.")

Final FlavorDB findings saved.


Final validation

In [76]:
flavordb_validation_df = pd.DataFrame([
    {
        "check": "Ingredient files loaded",
        "passed": len(flavordb_json_files) > 0,
    },
    {
        "check": "Molecule records created",
        "passed": len(flavordb_molecule_df) > 0,
    },
    {
        "check": "Taste descriptors extracted",
        "passed": len(flavordb_taste_long_df) > 0,
    },
    {
        "check": "Odor descriptors extracted",
        "passed": len(flavordb_odor_long_df) > 0,
    },
    {
        "check": "Ingredient profiles created",
        "passed": len(
            flavordb_ingredient_profile_df
        ) > 0,
    },
    {
        "check": "CeylonSavour mapping created",
        "passed": len(
            ceylonsavour_flavor_mapping_df
        ) > 0,
    },
])

display(flavordb_validation_df)

assert flavordb_validation_df["passed"].all(), (
    "One or more FlavorDB validation checks failed."
)

,check,passed
0,Ingredient files loaded,True
1,Molecule records created,True
2,Taste descriptors extracted,True
3,Odor descriptors extracted,True
4,Ingredient profiles created,True
5,CeylonSavour mapping created,True
